# MalariAI Revision Experiments (Issues 4, 5, 6)

Three self-contained experiments for the Array resubmission revision plan. Run sections in order (A, then B, then C) in one Kaggle session — GPU (T4) required.

- **Section A** — k-fold cross-validation on Stage 2 (EfficientNet-B0 classifier), Issue 4
- **Section B** — quantitative Grad-CAM++ localization metric, Issue 5
- **Section C** — U-Net segmentation baseline vs. Stage 1 watershed, Issue 6 (the long pole)

All three reuse the exact BBBC041 split/seed already used for Baseline A/B and Pipeline B (seed=42), so results are directly comparable to numbers already in the paper.

`DATA_ROOT` auto-detects your dataset's mount path in the first code cell by searching for a `malaria/training.json` file under `/kaggle/input` -- no editing needed unless auto-detect fails (it'll tell you if so).

**Version history if you're re-running this:** v1 had an I/O bug (re-decoding full images per crop, ~40 min/epoch -- would never finish). v2 fixed that but had a RAM leak (Python list-of-dicts metadata + never freeing memory between folds) that could OOM-crash mid-run. v3 (this version) fixes the RAM issue AND restores `num_workers>0` + drops unhelpful `DataParallel` overhead for the tiny classifier, which should cut v2's stable-but-slow ~237s/epoch down further. If you're mid-run on v2 and it's not crashing, it's safe to let it finish (~2 hrs/fold, ~10 hrs total worst case) -- restarting with v3 is only worth the reset if you'd rather it finish faster.

Download `/kaggle/working/phase6_outputs/*.json` when done and send them back — that's what gets written into the manuscript.

## Setup — paths, imports, shared definitions

This notebook is self-contained (no dependency on the original repo's file layout) so it runs standalone on Kaggle. The crop dataset materializes a pixel cache once here (~1-2 min) so every fold/epoch after that reads from RAM, not disk.

In [3]:
import os, sys, json, time, random, gc
from pathlib import Path
from collections import defaultdict

import numpy as np
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset
import torchvision.transforms as T
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
from PIL import Image
from tqdm.auto import tqdm

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

# ---- Kaggle paths -- auto-detected, no editing needed ----
# Searches every /kaggle/input/<dataset-slug>/ for a "malaria" folder that
# contains training.json (matches the BBBC041 layout: malaria/images/,
# malaria/training.json, malaria/test.json -- whatever slug Kaggle mounted
# the dataset under).
_candidates = list(Path("/kaggle/input").glob("*/malaria/training.json")) + \
              list(Path("/kaggle/input").glob("*/*/malaria/training.json")) + \
              list(Path("/kaggle/input").glob("malaria/training.json"))
if not _candidates:
    # fall back to a manual path if auto-detect fails -- edit this line
    DATA_ROOT = Path("/kaggle/input/datasets/kaysarulanas/bbbc041-malaria/malaria")
else:
    DATA_ROOT = _candidates[0].parent

TRAIN_JSON = DATA_ROOT / "training.json"
TEST_JSON  = DATA_ROOT / "test.json"
IMG_DIR    = DATA_ROOT / "images"
OUT_DIR    = Path("/kaggle/working/phase6_outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

assert TRAIN_JSON.exists(), (
    f"Not found: {TRAIN_JSON} -- auto-detect failed. Run "
    f"`!find /kaggle/input -maxdepth 3` in a cell to see the actual mounted "
    f"path, then set DATA_ROOT manually above."
)
print("Train JSON :", TRAIN_JSON)
print("Test JSON  :", TEST_JSON, "exists:", TEST_JSON.exists())
print("Image dir  :", IMG_DIR)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device, "--", torch.cuda.get_device_name(0) if device.type == "cuda" else "CPU")

Train JSON : /kaggle/input/datasets/kaysarulanas/bbbc041-malaria/malaria/training.json
Test JSON  : /kaggle/input/datasets/kaysarulanas/bbbc041-malaria/malaria/test.json exists: True
Image dir  : /kaggle/input/datasets/kaysarulanas/bbbc041-malaria/malaria/images
Device: cuda -- Tesla T4


In [4]:
# ---- shared/label_map.py, inlined ----
LABEL_TO_INT = {
    "background":     0,
    "red blood cell": 1,
    "trophozoite":    2,
    "ring":           3,
    "schizont":       4,
    "gametocyte":     5,
    "leukocyte":      6,
}
INT_TO_LABEL = {v: k for k, v in LABEL_TO_INT.items()}
NUM_CLASSES  = 7
PARASITE_CLASSES = ["trophozoite", "ring", "schizont", "gametocyte"]
SKIP_LABELS = {"difficult"}

print("Classes:", INT_TO_LABEL)

Classes: {0: 'background', 1: 'red blood cell', 2: 'trophozoite', 3: 'ring', 4: 'schizont', 5: 'gametocyte', 6: 'leukocyte'}


In [5]:
# ---- Phase1-EDA/dataset.py, inlined (MalariaCropDataset + MalariaDataset) ----
# Identical parsing/crop/margin logic to the original repo files -- same
# split/seed, so results here are comparable to everything already in the
# paper. materialize()/with_mode() and the numpy-array metadata backing are
# performance/memory fixes, not methodology changes -- see notes below.

def get_classification_transform(train: bool = True, img_size: int = 64):
    mean = [0.485, 0.456, 0.406]
    std  = [0.229, 0.224, 0.225]
    if train:
        return T.Compose([
            T.Resize((img_size, img_size)),
            T.RandomHorizontalFlip(p=0.5),
            T.RandomVerticalFlip(p=0.5),
            T.RandomRotation(degrees=30),
            T.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.05),
            T.ToTensor(),
            T.Normalize(mean, std),
        ])
    return T.Compose([T.Resize((img_size, img_size)), T.ToTensor(), T.Normalize(mean, std)])


class MalariaCropDataset(Dataset):
    """Per-cell crop dataset -- identical crop/margin logic to Phase1-EDA/dataset.py.

    Two performance/memory fixes on top of the original, neither of which
    changes what data the model actually sees:

    1. PIXEL CACHE (materialize()/with_mode()): the naive version re-opens
       and re-decodes the full 1600x1200 source image from disk for EVERY
       crop, every epoch (~79.7k redundant decodes/epoch vs ~1.2k). This
       made the first real Kaggle run take ~2400-2500s/epoch instead of the
       ~1-3 min/epoch this model/crop size should take on a T4.

    2. NUMPY-BACKED METADATA (this class no longer stores crops as a Python
       list of ~79.7k dicts). A plain list-of-dicts is a classic PyTorch
       DataLoader memory blowup with num_workers>0: every __getitem__ call
       touches Python object refcounts on the dict + its boxed int fields,
       and under fork-based multiprocessing that refcount WRITE triggers
       copy-on-write duplication of that memory page in every worker
       process. Over an epoch of shuffled access this ends up duplicating
       most of the crop metadata's memory per worker (a well-documented
       PyTorch/Python issue, e.g. pytorch/pytorch#13246). Storing box
       coordinates and label indices as flat numpy arrays (no per-element
       Python objects) avoids this -- indexing a numpy array doesn't touch
       per-element refcounts the way indexing a list of dicts does. This
       most likely caused (or contributed heavily to) the
       "tried to allocate more memory than is available" crash on the
       overnight run, on top of not freeing GPU/CPU memory between folds.
    """

    def __init__(self, json_path=None, image_dir=None, train: bool = True, img_size: int = 64,
                 margin: int = 6, _crop_arrays=None, _pixels=None):
        self.image_dir = Path(image_dir) if image_dir is not None else None
        self.train = train
        self.img_size = img_size
        self.margin = margin
        self.transform = get_classification_transform(train=train, img_size=img_size)
        if _crop_arrays is not None:
            self._img_name_ids, self._boxes, self._label_idx, self._image_names = _crop_arrays
            self._pixels = _pixels
        else:
            self._img_name_ids, self._boxes, self._label_idx, self._image_names = self._parse(Path(json_path))
            self._pixels = None

    def _parse(self, json_path):
        with open(json_path) as f:
            raw = json.load(f)
        image_names, name_to_id = [], {}
        img_name_ids, boxes, label_idx = [], [], []
        for item in raw:
            img_name = Path(item["image"]["pathname"]).name
            if img_name not in name_to_id:
                name_to_id[img_name] = len(image_names)
                image_names.append(img_name)
            nid = name_to_id[img_name]
            img_w = item["image"]["shape"]["c"]
            img_h = item["image"]["shape"]["r"]
            for obj in item.get("objects", []):
                label = obj["category"]
                if label in SKIP_LABELS or label not in LABEL_TO_INT:
                    continue
                bb = obj["bounding_box"]
                x_min = int(bb["minimum"]["c"]); y_min = int(bb["minimum"]["r"])
                x_max = int(bb["maximum"]["c"]); y_max = int(bb["maximum"]["r"])
                if x_max <= x_min or y_max <= y_min:
                    continue
                x0 = max(0, x_min - self.margin); y0 = max(0, y_min - self.margin)
                x1 = min(img_w, x_max + self.margin); y1 = min(img_h, y_max + self.margin)
                tx0, ty0 = max(0, x_min), max(0, y_min)
                tx1, ty1 = min(img_w, x_max), min(img_h, y_max)
                img_name_ids.append(nid)
                boxes.append((x0, y0, x1, y1, tx0, ty0, tx1, ty1))
                label_idx.append(LABEL_TO_INT[label])
        img_name_ids = np.asarray(img_name_ids, dtype=np.int32)
        boxes = np.asarray(boxes, dtype=np.int32)
        label_idx = np.asarray(label_idx, dtype=np.int64)
        return img_name_ids, boxes, label_idx, image_names

    def materialize(self):
        """Decode every crop once into a fixed-size uint8 pixel cache. Call
        this ONCE per JSON file (train or test), then reuse via with_mode()
        for eval-mode variants and Subset() for k-fold splits -- never
        re-instantiate from JSON mid-loop, that re-triggers a full decode."""
        if self._pixels is not None:
            return self
        by_image = defaultdict(list)
        for i, nid in enumerate(self._img_name_ids):
            by_image[int(nid)].append(i)

        n = len(self._label_idx)
        pixels = np.zeros((n, self.img_size, self.img_size, 3), dtype=np.uint8)
        for nid, idxs in tqdm(by_image.items(), desc="Materializing crop pixel cache"):
            img_name = self._image_names[nid]
            image = Image.open(self.image_dir / img_name).convert("RGB")
            for i in idxs:
                x0, y0, x1, y1 = self._boxes[i, 0:4].tolist()
                crop = image.crop((x0, y0, x1, y1)).resize((self.img_size, self.img_size))
                pixels[i] = np.array(crop)
            image.close()
        self._pixels = pixels
        print(f"  Pixel cache: {pixels.nbytes / 1e6:.0f} MB in RAM for {n:,} crops "
              f"from {len(by_image):,} source images.")
        return self

    def with_mode(self, train: bool):
        """Cheap variant sharing the same pixel cache/crop metadata but a
        different transform (train-time augmentation vs. eval-time none).
        Use this instead of re-instantiating from JSON."""
        new = MalariaCropDataset(
            train=train, img_size=self.img_size, margin=self.margin,
            _crop_arrays=(self._img_name_ids, self._boxes, self._label_idx, self._image_names),
            _pixels=self._pixels,
        )
        new.image_dir = self.image_dir
        return new

    def __len__(self):
        return len(self._label_idx)

    def __getitem__(self, idx):
        if self._pixels is not None:
            crop = Image.fromarray(self._pixels[idx])
        else:
            nid = int(self._img_name_ids[idx])
            x0, y0, x1, y1 = self._boxes[idx, 0:4].tolist()
            image = Image.open(self.image_dir / self._image_names[nid]).convert("RGB")
            crop = image.crop((x0, y0, x1, y1))
        crop_t = self.transform(crop)
        return crop_t, int(self._label_idx[idx])

    def get_inner_box_mask(self, idx):
        """Binary mask, shape [img_size, img_size], 1 where the *tight* GT box
        maps to after the crop was resized to img_size x img_size. Used by
        Section B to measure whether Grad-CAM++ energy concentrates on the
        cell itself vs. the surrounding margin."""
        x0, y0, x1, y1, tx0, ty0, tx1, ty1 = self._boxes[idx].tolist()
        cw, ch = (x1 - x0), (y1 - y0)
        if cw <= 0 or ch <= 0:
            return np.ones((self.img_size, self.img_size), dtype=np.float32)
        sx, sy = self.img_size / cw, self.img_size / ch
        ix0 = int(round((tx0 - x0) * sx)); iy0 = int(round((ty0 - y0) * sy))
        ix1 = int(round((tx1 - x0) * sx)); iy1 = int(round((ty1 - y0) * sy))
        ix0, iy0 = max(0, ix0), max(0, iy0)
        ix1, iy1 = min(self.img_size, ix1), min(self.img_size, iy1)
        mask = np.zeros((self.img_size, self.img_size), dtype=np.float32)
        if ix1 > ix0 and iy1 > iy0:
            mask[iy0:iy1, ix0:ix1] = 1.0
        return mask

    def label_of(self, idx):
        return INT_TO_LABEL[int(self._label_idx[idx])]

    def img_name_of(self, idx):
        return self._image_names[int(self._img_name_ids[idx])]


class MalariaDataset(Dataset):
    """Whole-image dataset -- identical to Phase1-EDA/dataset.py. Used here
    only to reproduce the exact same 966/242 (actually 967/241, int() truncation)
    image-level train/val split via the same random_split(seed=42) call, so
    Section C's U-Net trains/evaluates on the identical images as every other
    baseline in the paper. Not a bottleneck (only ~1207 records total, not
    79.7k) -- left as a plain list of dicts, negligible COW risk at this size."""

    def __init__(self, json_path, image_dir):
        self.image_dir = Path(image_dir)
        self._records = self._parse(Path(json_path))

    def _parse(self, json_path):
        with open(json_path) as f:
            raw = json.load(f)
        records = []
        for item in raw:
            img_name = Path(item["image"]["pathname"]).name
            boxes, labels = [], []
            for obj in item.get("objects", []):
                label = obj["category"]
                if label in SKIP_LABELS or label not in LABEL_TO_INT:
                    continue
                bb = obj["bounding_box"]
                x_min = float(bb["minimum"]["c"]); y_min = float(bb["minimum"]["r"])
                x_max = float(bb["maximum"]["c"]); y_max = float(bb["maximum"]["r"])
                if x_max <= x_min or y_max <= y_min:
                    continue
                boxes.append([x_min, y_min, x_max, y_max])
                labels.append(LABEL_TO_INT[label])
            if boxes:
                records.append({"img_name": img_name, "boxes": boxes, "labels": labels})
        return records

    def __len__(self):
        return len(self._records)

    def __getitem__(self, idx):
        return self._records[idx]


full_crop_ds = MalariaCropDataset(TRAIN_JSON, IMG_DIR, train=True)
print(f"Total labelled crops: {len(full_crop_ds):,}")
full_crop_ds.materialize()   # one-time decode pass -- every fold/epoch after this reads from RAM

full_img_ds = MalariaDataset(TRAIN_JSON, IMG_DIR)
n_val = int(len(full_img_ds) * 0.2)
n_train = len(full_img_ds) - n_val
train_img_idx, val_img_idx = torch.utils.data.random_split(
    range(len(full_img_ds)), [n_train, n_val], generator=torch.Generator().manual_seed(SEED)
)
train_img_names = {full_img_ds._records[i]["img_name"] for i in train_img_idx.indices}
val_img_names   = {full_img_ds._records[i]["img_name"] for i in val_img_idx.indices}
print(f"Image-level split: {len(train_img_names)} train / {len(val_img_names)} val "
      f"(same seed=42 split as Baseline A/B and Pipeline B)")


Total labelled crops: 79,672


Materializing crop pixel cache:   0%|          | 0/1208 [00:00<?, ?it/s]

  Pixel cache: 979 MB in RAM for 79,672 crops from 1,208 source images.
Image-level split: 967 train / 241 val (same seed=42 split as Baseline A/B and Pipeline B)


---
## Section A -- k-fold cross-validation on Stage 2 (Issue 4)

Wraps the existing Stage 2 training loop (EfficientNet-B0 + Focal Loss, identical hyperparameters to `Phase3-PipelineB/stage2_train.py`) in `sklearn.model_selection.StratifiedKFold` instead of a single random split. Reports mean +/- std accuracy across folds so the single-split 98.36% figure gets an actual confidence interval.

Uses **`StratifiedKFold`, not plain `KFold`** -- schizont (179), gametocyte (144), and leukocyte (103) crops are rare enough next to 77,420 RBC crops that a random split could leave a fold with too few (or zero) examples of a class. Stratifying on label keeps class proportions consistent across all 5 folds.

Default 5 folds x **up to 30 epochs, early-stopped after 6 epochs without improvement in the 3-epoch smoothed val accuracy** (matches the paper's original Stage 2 run, where the best epoch was 27/30; smoothed rather than raw val_acc because the first real run showed it swinging 90%->96%->91% epoch to epoch, which would either reset patience on noise or never trigger it). Uses `num_workers=4` with persistent workers (safe now that crop metadata is numpy-backed, not Python dicts) and skips `DataParallel` by default since it added overhead without benefit for a model/batch this small on the first real run (~237s/epoch on T4x2 despite the pixel cache fix). Each fold prints its raw and smoothed val_acc so you can see the trend, plus whether it plateaued or was still improving when it stopped.

In [6]:
from sklearn.model_selection import StratifiedKFold

_TRAIN_COUNTS = {1: 77420, 2: 1473, 3: 353, 4: 179, 5: 144, 6: 103}

def compute_focal_alpha(num_classes=NUM_CLASSES):
    total = sum(_TRAIN_COUNTS.values())
    alpha = [0.0]
    for i in range(1, num_classes):
        count = _TRAIN_COUNTS.get(i, 1)
        alpha.append(total / (len(_TRAIN_COUNTS) * count))
    s = sum(alpha[1:])
    alpha = [a / s * (num_classes - 1) for a in alpha]
    return torch.tensor(alpha, dtype=torch.float32)


class FocalLoss(nn.Module):
    def __init__(self, alpha, gamma=2.0):
        super().__init__()
        self.register_buffer("alpha", alpha)
        self.gamma = gamma

    def forward(self, logits, targets):
        ce = F.cross_entropy(logits, targets, reduction="none")
        pt = torch.exp(-ce)
        at = self.alpha[targets]
        return (at * (1.0 - pt) ** self.gamma * ce).mean()


def build_efficientnet_b0(num_classes=NUM_CLASSES, pretrained=True):
    weights = EfficientNet_B0_Weights.IMAGENET1K_V1 if pretrained else None
    model = efficientnet_b0(weights=weights)
    in_feat = model.classifier[1].in_features
    model.classifier[1] = nn.Linear(in_feat, num_classes)
    return model


def maybe_data_parallel(model):
    """NOT called anywhere in this notebook, and not recommended: a real run
    that used nn.DataParallel across a Kaggle T4x2 session's 2 GPUs hung
    completely after epoch 12 (confirmed -- 9+ hours with zero progress, same
    log line repeating). This is a known failure mode (multi-GPU scatter/
    gather deadlocking on P2P communication in Kaggle's container) on top of
    the overhead problem (a model this small on 64x64 crops doesn't have
    enough compute per batch to make splitting across 2 GPUs worth the sync
    cost anyway). Left defined only for reference -- do not call this."""
    if torch.cuda.device_count() > 1:
        return nn.DataParallel(model)
    return model


def make_loader(dataset, batch_size, shuffle, num_workers=4):
    """Shared DataLoader factory. num_workers>0 is safe again now that crop
    metadata is numpy-backed (see MalariaCropDataset docstring) -- the previous
    OOM was from a Python list-of-dicts, not from having workers per se. Workers
    let CPU-side augmentation (PIL crop/rotate/color-jitter) overlap with GPU
    compute instead of serializing with it, which is the main lever left after
    fixing the pixel-cache I/O and the RAM leak."""
    kwargs = dict(batch_size=batch_size, shuffle=shuffle, pin_memory=True)
    if num_workers > 0:
        kwargs.update(num_workers=num_workers, persistent_workers=True, prefetch_factor=4)
    else:
        kwargs["num_workers"] = 0
    return DataLoader(dataset, **kwargs)


def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for crops, labels in loader:
        crops, labels = crops.to(device), labels.to(device)
        logits = model(crops)
        loss = criterion(logits, labels)
        optimizer.zero_grad(); loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()
        total_loss += loss.item() * len(labels)
        correct += (logits.argmax(1) == labels).sum().item()
        total += len(labels)
    return total_loss / total, correct / total


@torch.no_grad()
def evaluate_fold(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    class_correct = {i: 0 for i in range(NUM_CLASSES)}
    class_total = {i: 0 for i in range(NUM_CLASSES)}
    for crops, labels in loader:
        crops, labels = crops.to(device), labels.to(device)
        logits = model(crops)
        loss = criterion(logits, labels)
        preds = logits.argmax(1)
        total_loss += loss.item() * len(labels)
        correct += (preds == labels).sum().item()
        total += len(labels)
        for lbl, pred in zip(labels.cpu(), preds.cpu()):
            class_total[lbl.item()] += 1
            class_correct[lbl.item()] += int(pred == lbl)
    per_class_acc = {INT_TO_LABEL[i]: round(class_correct[i] / class_total[i] * 100, 2)
                      for i in range(NUM_CLASSES) if class_total[i] > 0}
    return total_loss / total, correct / total, per_class_acc


N_FOLDS  = 3             # set to 3 (down from 5) given tight remaining GPU quota -- still
                         # a defensible k-fold CV. Bump back to 5 if you have quota to spare.
N_EPOCHS = 30            # matches the paper's original Stage 2 run (best epoch was 27/30)
EARLY_STOP_PATIENCE = 6  # stop a fold early if val_acc hasn't improved in this many epochs
BATCH    = 64
LR       = 1e-4

all_labels = np.array([full_crop_ds.label_of(i) for i in range(len(full_crop_ds))])
all_label_idx = np.array([LABEL_TO_INT[l] for l in all_labels])

# StratifiedKFold (not plain KFold): with schizont (179), gametocyte (144), and
# leukocyte (103) this rare relative to 77,420 RBCs, a random KFold can easily
# starve a fold of a whole class. Stratifying on label keeps class proportions
# consistent across all 5 folds/val splits.
kf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

# Built ONCE outside the fold loop -- previously this was rebuilt from JSON
# inside the loop on every fold, which (before the pixel-cache fix above)
# re-triggered a full disk decode 5 times over. Now it's a free shared view.
val_ds_noaug = full_crop_ds.with_mode(train=False)

# RESUME SUPPORT: nothing used to be saved until ALL folds finished -- if the
# kernel got killed by a GPU-quota/session-time limit mid-run, every completed
# fold's work was lost with it. Now every fold is saved to disk immediately
# after it finishes, and already-completed folds (matched by fold number) are
# skipped on re-run so you don't burn quota redoing them.
ckpt_path = OUT_DIR / "kfold_metrics.json"
if ckpt_path.exists():
    with open(ckpt_path) as f:
        fold_results = json.load(f).get("fold_results", [])
    completed_folds = {r["fold"] for r in fold_results}
    print(f"Resuming from {ckpt_path}: {len(completed_folds)} fold(s) already completed "
          f"({sorted(completed_folds)}) -- these will be skipped.")
else:
    fold_results, completed_folds = [], set()

t_start = time.time()

for fold, (train_idx, val_idx) in enumerate(kf.split(np.arange(len(full_crop_ds)), all_label_idx), start=1):
    if fold in completed_folds:
        continue
    print(f"\n{'='*60}\nFold {fold}/{N_FOLDS}  (train={len(train_idx)}  val={len(val_idx)})\n{'='*60}")

    train_subset = Subset(full_crop_ds, train_idx.tolist())
    val_subset = Subset(val_ds_noaug, val_idx.tolist())

    train_loader = make_loader(train_subset, BATCH, shuffle=True)
    val_loader   = make_loader(val_subset,   BATCH, shuffle=False)

    model = build_efficientnet_b0().to(device)  # no DataParallel here by default -- see note above
    alpha = compute_focal_alpha().to(device)
    criterion = FocalLoss(alpha=alpha, gamma=2.0)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=N_EPOCHS)

    best_val_acc, best_per_class, best_epoch, epochs_since_improve = 0.0, {}, 0, 0
    best_smoothed, val_acc_history = 0.0, []
    fold_start = time.time()
    for epoch in range(N_EPOCHS):
        tr_loss, tr_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
        vl_loss, vl_acc, per_class = evaluate_fold(model, val_loader, criterion, device)
        scheduler.step()
        val_acc_history.append(vl_acc)

        # Raw best (for reporting -- comparable to the paper's single-checkpoint number)
        if vl_acc > best_val_acc + 1e-4:
            best_val_acc, best_per_class, best_epoch = vl_acc, per_class, epoch + 1

        # Smoothed (3-epoch moving average) for the STOP decision -- val_acc swings
        # epoch to epoch (e.g. 90%->96%->91% seen in the first real run), so patience
        # on the raw number either resets on noise or never triggers. The smoothed
        # trend is what actually tells you whether the fold is still improving.
        smoothed = float(np.mean(val_acc_history[-3:]))
        if smoothed > best_smoothed + 1e-4:
            best_smoothed, epochs_since_improve = smoothed, 0
        else:
            epochs_since_improve += 1

        ep_time = time.time() - fold_start
        print(f"  epoch {epoch+1:02d}/{N_EPOCHS}  train_acc={tr_acc*100:.2f}%  val_acc={vl_acc*100:.2f}%  "
              f"(smoothed={smoothed*100:.2f}%, {ep_time/(epoch+1):.0f}s/epoch avg)")
        if epochs_since_improve >= EARLY_STOP_PATIENCE:
            print(f"  Early stop: smoothed val_acc hasn't improved in {EARLY_STOP_PATIENCE} epochs "
                  f"(best raw {best_val_acc*100:.2f}% at epoch {best_epoch}).")
            break

    n_epochs_run = epoch + 1
    plateaued = best_epoch <= n_epochs_run - 3 or n_epochs_run < N_EPOCHS
    print(f"  Fold {fold} best val acc: {best_val_acc*100:.2f}% (epoch {best_epoch}/{n_epochs_run} run)"
          f"  -- {'plateaued' if plateaued else 'STILL IMPROVING at cutoff, consider more epochs'}")
    fold_results.append({
        "fold": fold, "best_val_acc": round(best_val_acc * 100, 2), "best_epoch": best_epoch,
        "n_epochs_run": n_epochs_run, "plateaued": plateaued, "per_class_acc": best_per_class,
        "val_acc_history": [round(a * 100, 2) for a in val_acc_history],
    })

    # Explicit cleanup -- each fold builds a fresh model/optimizer/scheduler/loaders;
    # without deleting them, GPU memory and host RAM both accumulate fold over fold
    # instead of being reclaimed, which compounds the DataLoader-worker RAM issue above.
    del model, optimizer, scheduler, train_loader, val_loader, train_subset, val_subset
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    # Save after EVERY fold -- if this is the last one that finishes before the
    # session gets killed, you still have real numbers for however many folds
    # completed instead of nothing at all.
    _partial = {
        "n_folds": N_FOLDS, "n_epochs_cap": N_EPOCHS, "early_stop_patience": EARLY_STOP_PATIENCE,
        "kfold_strategy": "StratifiedKFold", "fold_results": fold_results,
        "n_folds_completed": len(fold_results),
        "status": "in_progress" if len(fold_results) < N_FOLDS else "complete",
    }
    with open(ckpt_path, "w") as f:
        json.dump(_partial, f, indent=2)
    print(f"  Checkpoint saved ({len(fold_results)}/{N_FOLDS} folds done) -> {ckpt_path}")

elapsed = time.time() - t_start
accs = np.array([r["best_val_acc"] for r in fold_results])
n_not_plateaued = sum(1 for r in fold_results if not r["plateaued"])

kfold_summary = {
    "n_folds": N_FOLDS, "n_epochs_cap": N_EPOCHS, "early_stop_patience": EARLY_STOP_PATIENCE,
    "kfold_strategy": "StratifiedKFold",
    "fold_results": fold_results,
    "n_folds_completed": len(fold_results),
    "status": "complete" if len(fold_results) == N_FOLDS else "PARTIAL -- ran out of folds before finishing",
    "mean_val_acc": round(float(accs.mean()), 2),
    "std_val_acc":  round(float(accs.std()), 2),
    "n_folds_not_plateaued": n_not_plateaued,
    "elapsed_seconds": round(elapsed, 1),
}

print(f"\n{'='*60}\nK-FOLD SUMMARY ({len(fold_results)}/{N_FOLDS} folds)\n{'='*60}")
print(f"  Accuracy across {len(fold_results)} fold(s): {kfold_summary['mean_val_acc']:.2f}% "
      f"+/- {kfold_summary['std_val_acc']:.2f}%")
print(f"  (single-split original figure was 98.36%, best epoch 27/30)")
if len(fold_results) < N_FOLDS:
    print(f"  NOTE: only {len(fold_results)}/{N_FOLDS} folds completed -- re-run this cell (it will "
          f"resume and skip completed folds) to finish before trusting this for the paper.")
if n_not_plateaued:
    print(f"  WARNING: {n_not_plateaued} completed fold(s) hit the {N_EPOCHS}-epoch cap while still "
          f"improving -- re-run with a higher N_EPOCHS before trusting this number for the paper.")

with open(ckpt_path, "w") as f:
    json.dump(kfold_summary, f, indent=2)
print(f"\nSaved -> {ckpt_path}")



Fold 1/3  (train=53114  val=26558)
Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 123MB/s] 


  epoch 01/30  train_acc=60.74%  val_acc=62.34%  (smoothed=62.34%, 63s/epoch avg)
  epoch 02/30  train_acc=81.56%  val_acc=82.43%  (smoothed=72.39%, 61s/epoch avg)
  epoch 03/30  train_acc=87.35%  val_acc=90.15%  (smoothed=78.31%, 61s/epoch avg)
  epoch 04/30  train_acc=89.06%  val_acc=88.26%  (smoothed=86.94%, 61s/epoch avg)
  epoch 05/30  train_acc=90.74%  val_acc=91.16%  (smoothed=89.85%, 60s/epoch avg)
  epoch 06/30  train_acc=91.30%  val_acc=91.03%  (smoothed=90.15%, 60s/epoch avg)
  epoch 07/30  train_acc=92.93%  val_acc=88.99%  (smoothed=90.40%, 60s/epoch avg)
  epoch 08/30  train_acc=92.64%  val_acc=90.77%  (smoothed=90.27%, 60s/epoch avg)
  epoch 09/30  train_acc=91.25%  val_acc=91.00%  (smoothed=90.25%, 60s/epoch avg)
  epoch 10/30  train_acc=92.48%  val_acc=95.64%  (smoothed=92.47%, 60s/epoch avg)
  epoch 11/30  train_acc=94.32%  val_acc=91.94%  (smoothed=92.86%, 60s/epoch avg)
  epoch 12/30  train_acc=93.65%  val_acc=94.40%  (smoothed=94.00%, 60s/epoch avg)
  epoch 13/30  t

Needs a trained Stage 2 classifier. If you have `best.pth` from the original training run, upload it as a Kaggle dataset/input and set `CHECKPOINT_PATH` below. **Otherwise this cell trains a fresh model on train-split crops only** (the same image-level split as everywhere else in the paper) -- never on the val-split crops this section then scores, to avoid train/eval leakage that would inflate the localization score for reasons unrelated to Grad-CAM++ actually working.

The metric also reports a **chance baseline**: the tight GT box's own area fraction of the 64x64 crop, computed for every crop regardless of correctness. Since the box already covers most of a tightly-margined (6px) crop, a high raw ratio alone doesn't prove anything -- `ratio_minus_chance` is the number that actually says whether Grad-CAM++ is concentrating better than the box geometry would give it for free.

In [7]:
# ---- Grad-CAM++ (Phase3-PipelineB/gradcam.py, inlined verbatim) ----
class GradCAMPlusPlus:
    def __init__(self, model, target_layer=None):
        self.model = model
        self.model.eval()
        self.target_layer = target_layer if target_layer is not None else model.features[-1]
        self._activations = None
        self._gradients = None
        self._fwd_hook = self.target_layer.register_forward_hook(self._save_activation)
        self._bwd_hook = self.target_layer.register_full_backward_hook(self._save_gradient)

    def _save_activation(self, module, inp, output):
        self._activations = output.detach()

    def _save_gradient(self, module, grad_input, grad_output):
        self._gradients = grad_output[0].detach()

    def remove_hooks(self):
        self._fwd_hook.remove(); self._bwd_hook.remove()

    def __call__(self, input_tensor, class_idx=None):
        dev = next(self.model.parameters()).device
        inp = input_tensor.to(dev).requires_grad_(True)
        logits = self.model(inp)
        probs = torch.softmax(logits, dim=1)
        pred_class = int(logits.argmax(1).item())
        confidence = float(probs[0, pred_class].item())
        target_cls = class_idx if class_idx is not None else pred_class
        self.model.zero_grad()
        logits[0, target_cls].backward()
        grads, acts = self._gradients, self._activations
        grads_sq, grads_cub = grads ** 2, grads ** 3
        numer = grads_sq
        denom = 2.0 * grads_sq + (acts * grads_cub).sum(dim=(2, 3), keepdim=True) + 1e-9
        alpha = numer / denom
        weights = (alpha * F.relu(grads)).sum(dim=(2, 3))
        cam = (weights[..., None, None] * acts).sum(dim=1, keepdim=True)
        cam = F.relu(cam)
        H, W = input_tensor.shape[-2:]
        cam_up = F.interpolate(cam, size=(H, W), mode="bilinear", align_corners=False)
        heatmap = cam_up.squeeze().detach().cpu().numpy()
        hmin, hmax = heatmap.min(), heatmap.max()
        heatmap = (heatmap - hmin) / (hmax - hmin) if hmax > hmin else np.zeros_like(heatmap)
        return heatmap.astype(np.float32), pred_class, confidence


CHECKPOINT_PATH = None  # e.g. "/kaggle/input/malariai-stage2-ckpt/best.pth" -- leave None to train fresh
# Caveat if you DO supply a checkpoint: the original stage2_train.py split crops
# randomly at the crop level (not by source image), so some crops from val_img_names
# images may have been in its training set. Fine for a qualitative check, but for a
# clean train/eval split prefer leaving this None so the split below is honored exactly.

if CHECKPOINT_PATH and Path(CHECKPOINT_PATH).exists():
    print(f"Loading existing checkpoint: {CHECKPOINT_PATH}")
    gc_model = build_efficientnet_b0(pretrained=False).to(device)
    ckpt = torch.load(CHECKPOINT_PATH, map_location=device)
    gc_model.load_state_dict(ckpt.get("model", ckpt))
else:
    # IMPORTANT: train only on crops from train_img_names, never on val_img_names.
    # Training on 100% of crops (including the images Section B evaluates
    # localization on below) would let the model memorize those exact cells --
    # train/eval leakage that inflates the energy-in-box ratio for reasons that
    # have nothing to do with Grad-CAM++ actually localizing well. This matches
    # the same image-level split used everywhere else in the paper. full_crop_ds
    # is already pixel-cached (materialized in Setup) so this Subset is free.
    print("No checkpoint provided -- training a fresh model on TRAIN-SPLIT crops only "
          "(same image-level split as everywhere else in the paper).")
    train_only_idx = [i for i in range(len(full_crop_ds))
                       if full_crop_ds.img_name_of(i) in train_img_names]
    print(f"  {len(train_only_idx)}/{len(full_crop_ds)} crops belong to train-split images.")
    train_only_subset = Subset(full_crop_ds, train_only_idx)

    gc_model = build_efficientnet_b0().to(device)  # no DataParallel -- see Section A note
    alpha = compute_focal_alpha().to(device)
    criterion = FocalLoss(alpha=alpha, gamma=2.0)
    optimizer = torch.optim.AdamW(gc_model.parameters(), lr=1e-4, weight_decay=1e-4)
    GC_EPOCHS, GC_PATIENCE = 30, 6
    gc_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=GC_EPOCHS)
    gc_loader = make_loader(train_only_subset, batch_size=64, shuffle=True)

    best_tr_acc, epochs_since_improve = 0.0, 0
    for epoch in range(GC_EPOCHS):
        tr_loss, tr_acc = train_one_epoch(gc_model, gc_loader, criterion, optimizer, device)
        gc_scheduler.step()
        print(f"  epoch {epoch+1:02d}/{GC_EPOCHS}  train_acc={tr_acc*100:.2f}%")
        if tr_acc > best_tr_acc + 1e-4:
            best_tr_acc, epochs_since_improve = tr_acc, 0
        else:
            epochs_since_improve += 1
        if epochs_since_improve >= GC_PATIENCE:
            print(f"  Early stop: train_acc hasn't improved in {GC_PATIENCE} epochs.")
            break

    del optimizer, gc_scheduler, gc_loader, train_only_subset
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

gc_model.eval()
cam = GradCAMPlusPlus(gc_model)
print("\nModel ready for Grad-CAM++ evaluation.")


No checkpoint provided -- training a fresh model on TRAIN-SPLIT crops only (same image-level split as everywhere else in the paper).
  63446/79672 crops belong to train-split images.
  epoch 01/30  train_acc=64.73%
  epoch 02/30  train_acc=84.17%
  epoch 03/30  train_acc=87.83%
  epoch 04/30  train_acc=89.05%
  epoch 05/30  train_acc=90.23%
  epoch 06/30  train_acc=90.97%
  epoch 07/30  train_acc=92.04%
  epoch 08/30  train_acc=91.10%
  epoch 09/30  train_acc=92.80%
  epoch 10/30  train_acc=93.08%
  epoch 11/30  train_acc=94.09%
  epoch 12/30  train_acc=94.86%
  epoch 13/30  train_acc=94.03%
  epoch 14/30  train_acc=95.41%
  epoch 15/30  train_acc=94.12%
  epoch 16/30  train_acc=95.73%
  epoch 17/30  train_acc=95.58%
  epoch 18/30  train_acc=95.74%
  epoch 19/30  train_acc=95.67%
  epoch 20/30  train_acc=96.06%
  epoch 21/30  train_acc=96.45%
  epoch 22/30  train_acc=96.45%
  epoch 23/30  train_acc=96.72%
  epoch 24/30  train_acc=97.16%
  epoch 25/30  train_acc=97.29%
  epoch 26/30  tr

In [8]:
# ---- Compute energy-in-box ratio across the validation-image parasite crops ----
val_ds_eval = full_crop_ds.with_mode(train=False)  # reuses the shared pixel cache,
                                                    # no re-decode from JSON/disk

# Restrict to crops whose *source image* is in the held-out val split (same
# image-level split as everything else in the paper), and whose GT label is
# a parasite class -- these are the clinically relevant crops Grad-CAM must
# localize correctly on.
parasite_idx = [i for i in range(len(val_ds_eval))
                if val_ds_eval.img_name_of(i) in val_img_names
                and val_ds_eval.label_of(i) in PARASITE_CLASSES]
print(f"Evaluating Grad-CAM++ localization on {len(parasite_idx)} held-out parasite-class crops.")

ratios_by_class = defaultdict(list)
chance_by_class = defaultdict(list)   # tight-box area fraction of the crop -- the "no-information" baseline
n_correct, n_total = 0, 0

for idx in tqdm(parasite_idx, desc="Grad-CAM++ localization"):
    crop_t, label_idx = val_ds_eval[idx]
    inp = crop_t.unsqueeze(0)
    heatmap, pred_class, conf = cam(inp)
    n_total += 1

    # Chance baseline is computed for every crop regardless of correctness --
    # it's a property of the box geometry, not the model.
    inner_mask = val_ds_eval.get_inner_box_mask(idx)
    chance_by_class[val_ds_eval.label_of(idx)].append(float(inner_mask.sum() / inner_mask.size))

    if pred_class != label_idx:
        continue  # only score the actual heatmap ratio on correctly-classified crops
    n_correct += 1

    total_energy = heatmap.sum()
    if total_energy <= 1e-8:
        continue
    in_box_energy = (heatmap * inner_mask).sum()
    ratio = float(in_box_energy / total_energy)
    ratios_by_class[val_ds_eval.label_of(idx)].append(ratio)

cam.remove_hooks()

summary = {}
all_ratios, all_chance = [], []
for cls in ratios_by_class:
    arr = np.array(ratios_by_class[cls])
    chance_arr = np.array(chance_by_class[cls])
    summary[cls] = {
        "n": len(arr),
        "mean_energy_in_box_ratio": round(float(arr.mean()), 4),
        "std": round(float(arr.std()), 4),
        "chance_baseline_area_fraction": round(float(chance_arr.mean()), 4),
        "ratio_minus_chance": round(float(arr.mean() - chance_arr.mean()), 4),
    }
    all_ratios.extend(ratios_by_class[cls])
    all_chance.extend(chance_by_class[cls])

overall = np.array(all_ratios)
overall_chance = np.array(all_chance)
gradcam_summary = {
    "n_crops_evaluated": n_total,
    "n_correctly_classified": n_correct,
    "overall_mean_energy_in_box_ratio": round(float(overall.mean()), 4) if len(overall) else None,
    "overall_std": round(float(overall.std()), 4) if len(overall) else None,
    "overall_chance_baseline_area_fraction": round(float(overall_chance.mean()), 4) if len(overall_chance) else None,
    "overall_ratio_minus_chance": round(float(overall.mean() - overall_chance.mean()), 4) if len(overall) else None,
    "per_class": summary,
    "metric_definition": (
        "Fraction of total Grad-CAM++ heatmap activation falling inside the "
        "tight (pre-margin) ground-truth box, evaluated only on correctly-"
        "classified parasite-class crops from the held-out validation images. "
        "1.0 = all activation on the cell; lower values indicate the "
        "explanation is spreading into the surrounding margin/background. "
        "chance_baseline_area_fraction is the tight box's own area fraction of "
        "the 64x64 crop (computed for every crop, not just correct ones) -- "
        "the score a spatially uninformative heatmap (e.g. uniform activation) "
        "would get by construction, since the box already covers most of a "
        "tightly-margined (6px) crop. Report ratio_minus_chance alongside the "
        "raw ratio so the metric can't be read as vacuous."
    ),
}

print(f"\n{'='*60}\nGRAD-CAM++ LOCALIZATION SUMMARY\n{'='*60}")
print(f"  Overall ratio  : {gradcam_summary['overall_mean_energy_in_box_ratio']:.4f} "
      f"+/- {gradcam_summary['overall_std']:.4f}  (n={n_correct} correctly-classified crops)")
print(f"  Chance baseline: {gradcam_summary['overall_chance_baseline_area_fraction']:.4f}")
print(f"  Ratio - chance : {gradcam_summary['overall_ratio_minus_chance']:+.4f}")
for cls, s in summary.items():
    print(f"  {cls:<14}: ratio={s['mean_energy_in_box_ratio']:.4f} +/- {s['std']:.4f}  "
          f"chance={s['chance_baseline_area_fraction']:.4f}  "
          f"(ratio-chance={s['ratio_minus_chance']:+.4f})  n={s['n']}")

with open(OUT_DIR / "gradcam_localization_metrics.json", "w") as f:
    json.dump(gradcam_summary, f, indent=2)
print(f"\nSaved -> {OUT_DIR / 'gradcam_localization_metrics.json'}")


Evaluating Grad-CAM++ localization on 429 held-out parasite-class crops.


Grad-CAM++ localization:   0%|          | 0/429 [00:00<?, ?it/s]


GRAD-CAM++ LOCALIZATION SUMMARY
  Overall ratio  : 0.8343 +/- 0.0219  (n=330 correctly-classified crops)
  Chance baseline: 0.8340
  Ratio - chance : +0.0003
  gametocyte    : ratio=0.8308 +/- 0.0218  chance=0.8311  (ratio-chance=-0.0003)  n=27
  ring          : ratio=0.8239 +/- 0.0129  chance=0.8239  (ratio-chance=+0.0000)  n=59
  trophozoite   : ratio=0.8359 +/- 0.0220  chance=0.8353  (ratio-chance=+0.0006)  n=221
  schizont      : ratio=0.8499 +/- 0.0254  chance=0.8482  (ratio-chance=+0.0017)  n=23

Saved -> /kaggle/working/phase6_outputs/gradcam_localization_metrics.json


---
## Section C -- U-Net segmentation baseline vs. Stage 1 watershed (Issue 6, the long pole)

BBBC041 has no pixel-level masks, only bounding boxes, so a supervised segmentation baseline needs synthetic training targets: this section fills an ellipse inscribed in each GT box to build a binary (cell vs. background) mask per training image -- standard practice in box-supervised cell-segmentation literature. Images are downsampled to 512x384 for U-Net training/inference (Kaggle T4 memory), then predicted masks are upsampled back to the original 1600x1200 resolution before evaluation so box coordinates stay comparable to Stage 1's watershed output.

Evaluation reuses the **exact same metric formulas** as Stage 1's `evaluate_stage1()` in `src/pipeline_b_v2/e2e_eval.py` (IoU@0.5 recall/precision/F1, centroid-in-box recall, biological-localization recall, infected-cell sensitivity) -- reimplemented here from scratch since the original function expects a pre-built CSV this notebook doesn't have, but the formulas are copied verbatim so the numbers are directly comparable to what's already in Table 5 of the paper. This includes Stage 1's `MAX_CELL_W = MAX_CELL_H = 220` oversized-box filter, applied to U-Net's predicted boxes exactly as it's applied to watershed's -- without it, U-Net blobs that merge adjacent RBCs into one giant connected component would drag precision down for a reason Stage 1 never has to face, since watershed's oversized outputs are filtered out before scoring too.

The point isn't to "win" against watershed -- it's to show where each approach wins: U-Net should have higher raw recall (it's supervised) but watershed needs zero annotations and produces organically-shaped regions that are more transparent in dense clusters.

In [9]:
# ---- Build binary segmentation masks from GT boxes (ellipse-filled) ----
DS_W, DS_H = 512, 384          # downsampled training/inference resolution
ORIG_W, ORIG_H = 1600, 1200    # BBBC041 native resolution
SCALE_X, SCALE_Y = DS_W / ORIG_W, DS_H / ORIG_H

def make_mask(boxes_1600, w=DS_W, h=DS_H):
    mask = np.zeros((h, w), dtype=np.uint8)
    for (x1, y1, x2, y2) in boxes_1600:
        cx = (x1 + x2) / 2 * SCALE_X
        cy = (y1 + y2) / 2 * SCALE_Y
        rx = max(1, (x2 - x1) / 2 * SCALE_X)
        ry = max(1, (y2 - y1) / 2 * SCALE_Y)
        cv2.ellipse(mask, (int(cx), int(cy)), (int(rx), int(ry)), 0, 0, 360, 1, -1)
    return mask


class UNetSegDataset(Dataset):
    def __init__(self, records, img_dir, img_names_filter, train=True):
        self.records = [r for r in records if r["img_name"] in img_names_filter]
        self.img_dir = Path(img_dir)
        self.train = train

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        rec = self.records[idx]
        img = cv2.imread(str(self.img_dir / rec["img_name"]))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (DS_W, DS_H))
        mask = make_mask(rec["boxes"])

        if self.train and random.random() < 0.5:
            img = np.fliplr(img).copy(); mask = np.fliplr(mask).copy()
        if self.train and random.random() < 0.5:
            img = np.flipud(img).copy(); mask = np.flipud(mask).copy()

        img_t = torch.from_numpy(img).permute(2, 0, 1).float() / 255.0
        mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
        std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
        img_t = (img_t - mean) / std
        mask_t = torch.from_numpy(mask).float().unsqueeze(0)
        return img_t, mask_t


unet_train_ds = UNetSegDataset(full_img_ds._records, IMG_DIR, train_img_names, train=True)
unet_val_ds   = UNetSegDataset(full_img_ds._records, IMG_DIR, val_img_names,   train=False)
print(f"U-Net train images: {len(unet_train_ds)}  val images: {len(unet_val_ds)}")

U-Net train images: 967  val images: 241


In [10]:
# ---- Small U-Net (from scratch, no external dependency) ----
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
        )
    def forward(self, x):
        return self.block(x)


class UNet(nn.Module):
    def __init__(self, in_ch=3, out_ch=1, base=32):
        super().__init__()
        self.enc1 = ConvBlock(in_ch, base)
        self.enc2 = ConvBlock(base, base * 2)
        self.enc3 = ConvBlock(base * 2, base * 4)
        self.enc4 = ConvBlock(base * 4, base * 8)
        self.pool = nn.MaxPool2d(2)
        self.bottleneck = ConvBlock(base * 8, base * 16)
        self.up4 = nn.ConvTranspose2d(base * 16, base * 8, 2, stride=2)
        self.dec4 = ConvBlock(base * 16, base * 8)
        self.up3 = nn.ConvTranspose2d(base * 8, base * 4, 2, stride=2)
        self.dec3 = ConvBlock(base * 8, base * 4)
        self.up2 = nn.ConvTranspose2d(base * 4, base * 2, 2, stride=2)
        self.dec2 = ConvBlock(base * 4, base * 2)
        self.up1 = nn.ConvTranspose2d(base * 2, base, 2, stride=2)
        self.dec1 = ConvBlock(base * 2, base)
        self.out_conv = nn.Conv2d(base, out_ch, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        e4 = self.enc4(self.pool(e3))
        b = self.bottleneck(self.pool(e4))
        d4 = self.dec4(torch.cat([self.up4(b), e4], dim=1))
        d3 = self.dec3(torch.cat([self.up3(d4), e3], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), e2], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))
        return self.out_conv(d1)


def dice_loss(logits, targets, eps=1e-6):
    probs = torch.sigmoid(logits)
    num = 2 * (probs * targets).sum(dim=(1, 2, 3)) + eps
    den = probs.sum(dim=(1, 2, 3)) + targets.sum(dim=(1, 2, 3)) + eps
    return 1 - (num / den).mean()


unet_model = UNet().to(device)
bce = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(unet_model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=25)

# This dataset is a plain list of ~1207 records (not the 79.7k-crop list that
# caused the OOM), so a couple of workers here is low-risk and helps overlap
# cv2.imread + resize with GPU compute -- reuses the same make_loader as A/B.
train_loader = make_loader(unet_train_ds, batch_size=8, shuffle=True, num_workers=2)
val_loader   = make_loader(unet_val_ds,   batch_size=8, shuffle=False, num_workers=2)

N_UNET_EPOCHS = 25
unet_train_losses = []

for epoch in range(N_UNET_EPOCHS):
    unet_model.train()
    total_loss = 0.0
    for imgs, masks in train_loader:
        imgs, masks = imgs.to(device), masks.to(device)
        logits = unet_model(imgs)
        loss = bce(logits, masks) + dice_loss(logits, masks)
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        total_loss += loss.item() * imgs.size(0)
    scheduler.step()
    avg_loss = total_loss / len(unet_train_ds)
    unet_train_losses.append(avg_loss)
    print(f"  epoch {epoch+1:02d}/{N_UNET_EPOCHS}  loss={avg_loss:.4f}")

torch.save(unet_model.state_dict(), OUT_DIR / "unet_stage1_baseline.pth")
print(f"\nU-Net trained. Checkpoint saved -> {OUT_DIR / 'unet_stage1_baseline.pth'}")

del optimizer, scheduler, train_loader, val_loader
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

  epoch 01/25  loss=0.6235
  epoch 02/25  loss=0.3071
  epoch 03/25  loss=0.2243
  epoch 04/25  loss=0.2007
  epoch 05/25  loss=0.1851
  epoch 06/25  loss=0.1768
  epoch 07/25  loss=0.1723
  epoch 08/25  loss=0.1700
  epoch 09/25  loss=0.1641
  epoch 10/25  loss=0.1628
  epoch 11/25  loss=0.1579
  epoch 12/25  loss=0.1553
  epoch 13/25  loss=0.1545
  epoch 14/25  loss=0.1520
  epoch 15/25  loss=0.1503
  epoch 16/25  loss=0.1483
  epoch 17/25  loss=0.1456
  epoch 18/25  loss=0.1437
  epoch 19/25  loss=0.1436
  epoch 20/25  loss=0.1411
  epoch 21/25  loss=0.1390
  epoch 22/25  loss=0.1376
  epoch 23/25  loss=0.1371
  epoch 24/25  loss=0.1364
  epoch 25/25  loss=0.1358

U-Net trained. Checkpoint saved -> /kaggle/working/phase6_outputs/unet_stage1_baseline.pth


In [11]:
# ---- U-Net inference -> connected-components -> boxes, upsampled to 1600x1200 ----
MIN_AREA_DS = 20   # pixel-area threshold at DOWNSAMPLED resolution (tune if too noisy/sparse)

@torch.no_grad()
def unet_predict_boxes(img_bgr):
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    img_ds = cv2.resize(img_rgb, (DS_W, DS_H))
    img_t = torch.from_numpy(img_ds).permute(2, 0, 1).float().unsqueeze(0) / 255.0
    mean = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)
    img_t = ((img_t - mean) / std).to(device)

    logits = unet_model(img_t)
    prob = torch.sigmoid(logits)[0, 0].cpu().numpy()
    binary = (prob > 0.5).astype(np.uint8)

    n_labels, labels_im, stats, _ = cv2.connectedComponentsWithStats(binary, connectivity=8)
    boxes = []
    for i in range(1, n_labels):  # skip background label 0
        area = stats[i, cv2.CC_STAT_AREA]
        if area < MIN_AREA_DS:
            continue
        x, y, w, h = stats[i, cv2.CC_STAT_LEFT], stats[i, cv2.CC_STAT_TOP], \
                     stats[i, cv2.CC_STAT_WIDTH], stats[i, cv2.CC_STAT_HEIGHT]
        # upsample box back to native 1600x1200 coordinates
        x1 = x / SCALE_X; y1 = y / SCALE_Y
        x2 = (x + w) / SCALE_X; y2 = (y + h) / SCALE_Y
        boxes.append((x1, y1, x2, y2))
    return boxes


# ---- Metric formulas, copied verbatim from src/pipeline_b_v2/e2e_eval.py ----
def iou(a, b):
    ix1, iy1 = max(a[0], b[0]), max(a[1], b[1])
    ix2, iy2 = min(a[2], b[2]), min(a[3], b[3])
    inter = max(0, ix2 - ix1) * max(0, iy2 - iy1)
    area_a = (a[2] - a[0]) * (a[3] - a[1])
    area_b = (b[2] - b[0]) * (b[3] - b[1])
    union = area_a + area_b - inter
    return inter / union if union > 0 else 0.0

def box_centroid(box):
    return (box[0] + box[2]) / 2.0, (box[1] + box[3]) / 2.0

def centroid_in_box(wb, gb):
    cx, cy = box_centroid(wb)
    return gb[0] <= cx <= gb[2] and gb[1] <= cy <= gb[3]

# Same guard Stage 1 applies after _run_stage1() in src/pipeline_b_v2/e2e_eval.py
# (MAX_CELL_W = MAX_CELL_H = 220, filters merged-blob detections before scoring).
# Applying it identically here so U-Net isn't penalized on precision for a
# reason that has nothing to do with segmentation quality -- without it, any
# U-Net blob that merges adjacent RBCs into one giant connected component would
# count as one (obviously wrong) prediction and drag precision down for a
# reason Stage 1 never has to deal with, since it's filtered out there too.
MAX_CELL_W = 220
MAX_CELL_H = 220

def is_oversized(box):
    x1, y1, x2, y2 = box
    return (x2 - x1) > MAX_CELL_W or (y2 - y1) > MAX_CELL_H

def box_diagonal(box):
    w, h = box[2] - box[0], box[3] - box[1]
    return (w ** 2 + h ** 2) ** 0.5

def biological_localization_match(wb, gb, fraction=0.5):
    cx_w, cy_w = box_centroid(wb); cx_g, cy_g = box_centroid(gb)
    diag = box_diagonal(gb)
    dist = ((cx_w - cx_g) ** 2 + (cy_w - cy_g) ** 2) ** 0.5
    return dist <= fraction * diag


PARASITE_SET = set(PARASITE_CLASSES)
RELAXED_THRESHOLDS = [0.25, 0.30, 0.50]
BIO_FRACTION = 0.50

val_records = [r for r in full_img_ds._records if r["img_name"] in val_img_names]

total_gt, total_pred = 0, 0
tp_at = {t: 0 for t in RELAXED_THRESHOLDS}
inf_tp_at = {t: 0 for t in RELAXED_THRESHOLDS}
total_inf_gt = 0
centroid_tp, bio_loc_tp = 0, 0

for rec in tqdm(val_records, desc="U-Net Stage-1 eval"):
    img_path = IMG_DIR / rec["img_name"]
    bgr = cv2.imread(str(img_path))
    if bgr is None:
        continue
    gt_boxes = [tuple(b) for b in rec["boxes"]]
    gt_labels = [INT_TO_LABEL[l] for l in rec["labels"]]

    pred_boxes = unet_predict_boxes(bgr)
    pred_boxes = [b for b in pred_boxes if not is_oversized(b)]
    total_gt += len(gt_boxes)
    total_pred += len(pred_boxes)
    total_inf_gt += sum(1 for l in gt_labels if l in PARASITE_SET)

    for thr in RELAXED_THRESHOLDS:
        gt_matched = [False] * len(gt_boxes)
        inf_tp_img = 0
        for pb in pred_boxes:
            best_iou, best_j = 0.0, -1
            for j, gb in enumerate(gt_boxes):
                if gt_matched[j]:
                    continue
                v = iou(pb, gb)
                if v > best_iou:
                    best_iou, best_j = v, j
            if best_iou >= thr and best_j >= 0:
                gt_matched[best_j] = True
                if gt_labels[best_j] in PARASITE_SET:
                    inf_tp_img += 1
                if thr == 0.50:
                    pass
                tp_at[thr] += 1
        inf_tp_at[thr] += inf_tp_img

    gt_c_matched = [False] * len(gt_boxes)
    for pb in pred_boxes:
        for j, gb in enumerate(gt_boxes):
            if gt_c_matched[j]:
                continue
            if centroid_in_box(pb, gb):
                gt_c_matched[j] = True; centroid_tp += 1; break

    gt_b_matched = [False] * len(gt_boxes)
    for pb in pred_boxes:
        for j, gb in enumerate(gt_boxes):
            if gt_b_matched[j]:
                continue
            if biological_localization_match(pb, gb, BIO_FRACTION):
                gt_b_matched[j] = True; bio_loc_tp += 1; break

recall50 = tp_at[0.50] / total_gt if total_gt else 0.0
prec50 = tp_at[0.50] / total_pred if total_pred else 0.0
f1_50 = (2 * prec50 * recall50 / (prec50 + recall50)) if (prec50 + recall50) > 0 else 0.0

unet_results = {
    "model": "U-Net (from-scratch, box-derived ellipse masks, 512x384 downsampled)",
    "oversized_filter_applied": f"MAX_CELL_W={MAX_CELL_W}, MAX_CELL_H={MAX_CELL_H} (same as Stage 1's is_oversized)",
    "total_gt_cells": total_gt,
    "total_predicted_boxes": total_pred,
    "recall_at_iou50": round(recall50, 4),
    "precision_at_iou50": round(prec50, 4),
    "f1_at_iou50": round(f1_50, 4),
    "recall_at_relaxed_iou": {f"iou{int(t*100):02d}": round(tp_at[t] / total_gt, 4) if total_gt else 0.0
                               for t in RELAXED_THRESHOLDS},
    "centroid_in_box_recall": round(centroid_tp / total_gt, 4) if total_gt else 0.0,
    "bio_localization_recall": round(bio_loc_tp / total_gt, 4) if total_gt else 0.0,
    "infected_cell_sensitivity": {f"iou{int(t*100):02d}": round(inf_tp_at[t] / total_inf_gt, 4) if total_inf_gt else 0.0
                                  for t in RELAXED_THRESHOLDS},
    "total_infected_gt": total_inf_gt,
}

print(f"\n{'='*60}\nU-NET BASELINE RESULTS (compare against Table 5 -- Stage 1 watershed)\n{'='*60}")
print(f"  Recall @IoU0.5     : {unet_results['recall_at_iou50']:.4f}")
print(f"  Precision @IoU0.5  : {unet_results['precision_at_iou50']:.4f}")
print(f"  F1 @IoU0.5         : {unet_results['f1_at_iou50']:.4f}")
print(f"  Centroid-in-box    : {unet_results['centroid_in_box_recall']:.4f}")
print(f"  Bio-localization   : {unet_results['bio_localization_recall']:.4f}")
print(f"  Infected sens.@0.5 : {unet_results['infected_cell_sensitivity']['iou50']:.4f}")

with open(OUT_DIR / "unet_stage1_metrics.json", "w") as f:
    json.dump(unet_results, f, indent=2)
print(f"\nSaved -> {OUT_DIR / 'unet_stage1_metrics.json'}")


U-Net Stage-1 eval:   0%|          | 0/241 [00:00<?, ?it/s]


U-NET BASELINE RESULTS (compare against Table 5 -- Stage 1 watershed)
  Recall @IoU0.5     : 0.2828
  Precision @IoU0.5  : 0.8135
  F1 @IoU0.5         : 0.4196
  Centroid-in-box    : 0.3356
  Bio-localization   : 0.3367
  Infected sens.@0.5 : 0.2051

Saved -> /kaggle/working/phase6_outputs/unet_stage1_metrics.json


---
## Done -- download these 3 files

```
/kaggle/working/phase6_outputs/kfold_metrics.json
/kaggle/working/phase6_outputs/gradcam_localization_metrics.json
/kaggle/working/phase6_outputs/unet_stage1_metrics.json
```

Send them back and the paper (Issues 4, 5, 6, plus the "final numbers verification" pass) gets written from the real numbers instead of placeholders.

---
## Section B (continued) -- fixing the metric instead of reframing around it

The energy-in-box ratio (0.8343) landed within noise of the chance baseline (0.8340). Root cause,
not just bad luck: `GradCAMPlusPlus` defaulted to hooking `model.features[-1]` -- EfficientNet-B0's
final block, following the standard "always use the last conv layer" Grad-CAM advice. That advice
assumes a 224x224 ImageNet input; on our 64x64 crops, the same stride pattern collapses the final
block down to roughly a 2x2 feature map (confirmed empirically below) -- only 4 distinguishable
regions, nowhere near enough spatial resolution to tell "inside the tight box" from "in the 6px
margin" anywhere in a 64x64 crop. A heatmap built from a 2x2 grid is necessarily close to spatially
uniform once upsampled, which is exactly why it tracked the area-fraction chance baseline instead of
beating it.

This is a genuine, fixable methodology mismatch, not a fundamental limit of Grad-CAM++ or of this
model -- so instead of reframing a weak number, the cell below tests earlier, higher-resolution
blocks (still deep enough to carry class-discriminative signal) using the **same already-trained
classifier, no retraining needed**, and reports real ratio-vs-chance numbers for each so the layer
choice is picked on evidence, not on whichever number is biggest.


In [12]:
# ---- Empirically check spatial resolution of candidate target layers ----
# (measured directly rather than hand-derived from EfficientNet-B0's stride
# table, so this is correct regardless of any assumptions about the exact
# architecture -- what matters is the real activation shape at 64x64 input.)
_dummy = torch.zeros(1, 3, 64, 64).to(device)

def _spatial_shape(layer):
    shapes = {}
    def _hook(m, i, o):
        shapes["hw"] = tuple(o.shape[-2:])
    h = layer.register_forward_hook(_hook)
    with torch.no_grad():
        gc_model(_dummy)
    h.remove()
    return shapes["hw"]

candidate_layers = {
    "features[-1] (default, final block)": gc_model.features[-1],
    "features[5]": gc_model.features[5],
    "features[4]": gc_model.features[4],
    "features[3]": gc_model.features[3],
    "features[2]": gc_model.features[2],
}

print("Spatial resolution at 64x64 input, per candidate layer:")
for name, layer in candidate_layers.items():
    print(f"  {name:<35}: {_spatial_shape(layer)}")

# ---- Recompute the energy-in-box ratio for each candidate layer ----
# Reuses full_crop_ds/val_ds_eval/parasite_idx/gc_model already in memory from
# the cells above -- no retraining, just re-running inference with a
# different Grad-CAM++ hook point.
print(f"\n{'='*60}\nLAYER ABLATION: ratio vs. chance per candidate target layer\n{'='*60}")
layer_ablation = {}

for name, layer in candidate_layers.items():
    cam_test = GradCAMPlusPlus(gc_model, target_layer=layer)
    ratios, chances = [], []
    for idx in tqdm(parasite_idx, desc=name, leave=False):
        crop_t, label_idx = val_ds_eval[idx]
        inp = crop_t.unsqueeze(0)
        heatmap, pred_class, conf = cam_test(inp)
        if pred_class != label_idx:
            continue
        inner_mask = val_ds_eval.get_inner_box_mask(idx)
        total_energy = heatmap.sum()
        if total_energy <= 1e-8:
            continue
        ratios.append(float((heatmap * inner_mask).sum() / total_energy))
        chances.append(float(inner_mask.sum() / inner_mask.size))
    cam_test.remove_hooks()

    r, c = np.array(ratios), np.array(chances)
    entry = {
        "spatial_shape": list(_spatial_shape(layer)),
        "n": len(r),
        "mean_ratio": round(float(r.mean()), 4),
        "std": round(float(r.std()), 4),
        "mean_chance": round(float(c.mean()), 4),
        "ratio_minus_chance": round(float(r.mean() - c.mean()), 4),
        # how many std's the improvement is away from zero -- the real test of
        # whether this is a genuine effect or still just noise
        "effect_size_in_stds": round(float((r.mean() - c.mean()) / (r.std() + 1e-9)), 2),
    }
    layer_ablation[name] = entry
    print(f"  {name:<35}: ratio={entry['mean_ratio']:.4f} +/- {entry['std']:.4f}  "
          f"chance={entry['mean_chance']:.4f}  ratio-chance={entry['ratio_minus_chance']:+.4f}  "
          f"({entry['effect_size_in_stds']:+.2f} std)")

with open(OUT_DIR / "gradcam_layer_ablation.json", "w") as f:
    json.dump(layer_ablation, f, indent=2)
print(f"\nSaved -> {OUT_DIR / 'gradcam_layer_ablation.json'}")

best_name = max(layer_ablation, key=lambda k: layer_ablation[k]["ratio_minus_chance"])
print(f"\nBest candidate: {best_name} (ratio-chance={layer_ablation[best_name]['ratio_minus_chance']:+.4f}, "
      f"{layer_ablation[best_name]['effect_size_in_stds']:+.2f} std)")
print("If this shows a real multi-std improvement, re-run Section B's main metric cell with "
      "GradCAMPlusPlus(gc_model, target_layer=<that layer>) as the new default, and use this "
      "layer choice + its empirical justification (spatial resolution match for 64x64 inputs) "
      "in the manuscript instead of features[-1].")


Spatial resolution at 64x64 input, per candidate layer:
  features[-1] (default, final block): (2, 2)
  features[5]                        : (4, 4)
  features[4]                        : (4, 4)
  features[3]                        : (8, 8)
  features[2]                        : (16, 16)

LAYER ABLATION: ratio vs. chance per candidate target layer


features[-1] (default, final block):   0%|          | 0/429 [00:00<?, ?it/s]

  features[-1] (default, final block): ratio=0.8343 +/- 0.0219  chance=0.8343  ratio-chance=-0.0000  (-0.00 std)


features[5]:   0%|          | 0/429 [00:00<?, ?it/s]

  features[5]                        : ratio=0.8831 +/- 0.0665  chance=0.8346  ratio-chance=+0.0485  (+0.73 std)


...

  features[4]                        : ratio=0.8133 +/- 0.0673  chance=0.8340  ratio-chance=-0.0208  (-0.31 std)


features[3]:   0%|          | 0/429 [00:00<?, ?it/s]

  features[3]                        : ratio=0.7902 +/- 0.0878  chance=0.8343  ratio-chance=-0.0441  (-0.50 std)


features[2]:   0%|          | 0/429 [00:00<?, ?it/s]

  features[2]                        : ratio=0.8224 +/- 0.1060  chance=0.8343  ratio-chance=-0.0119  (-0.11 std)

Saved -> /kaggle/working/phase6_outputs/gradcam_layer_ablation.json

Best candidate: features[5] (ratio-chance=+0.0485, +0.73 std)
If this shows a real multi-std improvement, re-run Section B's main metric cell with GradCAMPlusPlus(gc_model, target_layer=<that layer>) as the new default, and use this layer choice + its empirical justification (spatial resolution match for 64x64 inputs) in the manuscript instead of features[-1].


---
## Appendix -- consolidated results so far

Reads back whatever's been saved to `/kaggle/working/phase6_outputs/` up to this point (works
regardless of which sections have actually finished -- missing files are just skipped) and prints
one consolidated summary. Meant as a quick reference while writing up the paper, and as a sanity
check that the numbers you're about to copy into the manuscript match what's actually on disk.


In [13]:
# ---- Appendix: consolidated summary of everything saved so far ----
_result_files = {
    "Section A -- k-fold CV (Issue 4)":            "kfold_metrics.json",
    "Section B -- Grad-CAM++ localization (Issue 5)": "gradcam_localization_metrics.json",
    "Section B -- layer ablation":                  "gradcam_layer_ablation.json",
    "Section C -- U-Net baseline (Issue 6)":        "unet_stage1_metrics.json",
}

print(f"{'='*70}\nAPPENDIX -- RESULTS SNAPSHOT ({time.strftime('%Y-%m-%d %H:%M:%S')})\n{'='*70}\n")

_appendix_summary = {}
for title, fname in _result_files.items():
    fpath = OUT_DIR / fname
    print(f"--- {title} ---")
    if not fpath.exists():
        print(f"  (not run yet -- {fname} not found)\n")
        _appendix_summary[title] = None
        continue
    with open(fpath) as f:
        data = json.load(f)
    _appendix_summary[title] = data

    if fname == "kfold_metrics.json":
        n_done = data.get("n_folds_completed", len(data.get("fold_results", [])))
        n_total = data.get("n_folds", "?")
        print(f"  {n_done}/{n_total} folds completed. Status: {data.get('status', 'unknown')}")
        print(f"  Accuracy: {data.get('mean_val_acc')}% +/- {data.get('std_val_acc')}%  "
              f"(single-split original: 98.36%)")
        for r in data.get("fold_results", []):
            print(f"    fold {r['fold']}: {r['best_val_acc']}% (epoch {r.get('best_epoch','?')}, "
                  f"plateaued={r.get('plateaued','?')})")

    elif fname == "gradcam_localization_metrics.json":
        print(f"  Overall ratio: {data.get('overall_mean_energy_in_box_ratio')} "
              f"+/- {data.get('overall_std')}  |  chance: {data.get('overall_chance_baseline_area_fraction')}  "
              f"|  ratio-chance: {data.get('overall_ratio_minus_chance')}")
        for cls, s in data.get("per_class", {}).items():
            print(f"    {cls}: ratio={s['mean_energy_in_box_ratio']} chance={s['chance_baseline_area_fraction']} "
                  f"(diff={s['ratio_minus_chance']:+.4f}) n={s['n']}")

    elif fname == "gradcam_layer_ablation.json":
        for layer_name, s in data.items():
            print(f"    {layer_name:<35}: ratio={s['mean_ratio']} chance={s['mean_chance']} "
                  f"diff={s['ratio_minus_chance']:+.4f} ({s['effect_size_in_stds']:+.2f} std)  "
                  f"shape={s['spatial_shape']}")

    elif fname == "unet_stage1_metrics.json":
        print(f"  Recall@IoU0.5: {data.get('recall_at_iou50')}  Precision@IoU0.5: {data.get('precision_at_iou50')}  "
              f"F1@IoU0.5: {data.get('f1_at_iou50')}")
        print(f"  Centroid-in-box recall: {data.get('centroid_in_box_recall')}  "
              f"Bio-localization recall: {data.get('bio_localization_recall')}")
        print(f"  Infected-cell sensitivity@0.5: {data.get('infected_cell_sensitivity', {}).get('iou50')}")

    print()

with open(OUT_DIR / "appendix_summary.json", "w") as f:
    json.dump(_appendix_summary, f, indent=2)
print(f"Saved consolidated snapshot -> {OUT_DIR / 'appendix_summary.json'}")


APPENDIX -- RESULTS SNAPSHOT (2026-08-01 15:25:17)

--- Section A -- k-fold CV (Issue 4) ---
  3/3 folds completed. Status: complete
  Accuracy: 97.34% +/- 0.18%  (single-split original: 98.36%)
    fold 1: 97.43% (epoch 28, plateaued=False)
    fold 2: 97.09% (epoch 12, plateaued=True)
    fold 3: 97.51% (epoch 30, plateaued=False)

--- Section B -- Grad-CAM++ localization (Issue 5) ---
  Overall ratio: 0.8343 +/- 0.0219  |  chance: 0.834  |  ratio-chance: 0.0003
    gametocyte: ratio=0.8308 chance=0.8311 (diff=-0.0003) n=27
    ring: ratio=0.8239 chance=0.8239 (diff=+0.0000) n=59
    trophozoite: ratio=0.8359 chance=0.8353 (diff=+0.0006) n=221
    schizont: ratio=0.8499 chance=0.8482 (diff=+0.0017) n=23

--- Section B -- layer ablation ---
    features[-1] (default, final block): ratio=0.8343 chance=0.8343 diff=-0.0000 (-0.00 std)  shape=[2, 2]
    features[5]                        : ratio=0.8831 chance=0.8346 diff=+0.0485 (+0.73 std)  shape=[4, 4]
    features[4]                   

---
## Finalize -- adopt features[5], add a proper paired significance test

The ablation's "+/- std" comparison was a quick heuristic, not a real statistical test. Ratio and
chance are computed on the *same* crop, so they're correlated -- a **paired** test (t-test and
Wilcoxon signed-rank, both reported since Wilcoxon doesn't assume normality) controls for that
properly and is the number that actually belongs in the manuscript, not the rougher effect-size
estimate from the ablation cell. This re-runs the metric once more with `target_layer=features[5]`
(the ablation winner) and writes the authoritative result to `gradcam_localization_metrics_final.json`.


In [14]:
# ---- Finalize: features[5] + a proper paired significance test ----
from scipy import stats

cam_final = GradCAMPlusPlus(gc_model, target_layer=gc_model.features[5])

final_ratios_by_class = defaultdict(list)
final_chance_by_class = defaultdict(list)
n_correct_final, n_total_final = 0, 0

for idx in tqdm(parasite_idx, desc="Final metric (features[5])"):
    crop_t, label_idx = val_ds_eval[idx]
    inp = crop_t.unsqueeze(0)
    heatmap, pred_class, conf = cam_final(inp)
    n_total_final += 1
    if pred_class != label_idx:
        continue
    n_correct_final += 1
    inner_mask = val_ds_eval.get_inner_box_mask(idx)
    total_energy = heatmap.sum()
    if total_energy <= 1e-8:
        continue
    ratio = float((heatmap * inner_mask).sum() / total_energy)
    chance = float(inner_mask.sum() / inner_mask.size)
    final_ratios_by_class[val_ds_eval.label_of(idx)].append(ratio)
    final_chance_by_class[val_ds_eval.label_of(idx)].append(chance)

cam_final.remove_hooks()

all_ratios_f = np.array([r for v in final_ratios_by_class.values() for r in v])
all_chance_f = np.array([c for v in final_chance_by_class.values() for c in v])

# Paired tests: same crop contributes one ratio and one chance value, so pair them
# rather than treating the two groups as independent samples.
t_stat, t_p = stats.ttest_rel(all_ratios_f, all_chance_f)
try:
    w_stat, w_p = stats.wilcoxon(all_ratios_f, all_chance_f)
except ValueError:
    w_stat, w_p = None, None

per_class_final = {}
for cls in final_ratios_by_class:
    r_c, c_c = np.array(final_ratios_by_class[cls]), np.array(final_chance_by_class[cls])
    ct, cp = stats.ttest_rel(r_c, c_c) if len(r_c) > 1 else (None, None)
    per_class_final[cls] = {
        "n": len(r_c),
        "mean_ratio": round(float(r_c.mean()), 4), "std": round(float(r_c.std()), 4),
        "mean_chance": round(float(c_c.mean()), 4),
        "ratio_minus_chance": round(float(r_c.mean() - c_c.mean()), 4),
        "paired_ttest_p": (round(float(cp), 4) if cp is not None else None),
    }

final_summary = {
    "target_layer": "features[5] (selected via ablation -- see gradcam_layer_ablation.json)",
    "n_crops_evaluated": n_total_final,
    "n_correctly_classified": n_correct_final,
    "overall_mean_energy_in_box_ratio": round(float(all_ratios_f.mean()), 4),
    "overall_std": round(float(all_ratios_f.std()), 4),
    "overall_chance_baseline_area_fraction": round(float(all_chance_f.mean()), 4),
    "overall_ratio_minus_chance": round(float((all_ratios_f - all_chance_f).mean()), 4),
    "paired_ttest": {"t_stat": round(float(t_stat), 4), "p_value": float(t_p)},
    "wilcoxon_signed_rank": {
        "stat": (round(float(w_stat), 4) if w_stat is not None else None),
        "p_value": (float(w_p) if w_p is not None else None),
    },
    "per_class": per_class_final,
    "previous_default_layer_result_for_reference": {
        "target_layer": "features[-1] (original default, pre-ablation)",
        "ratio_minus_chance": 0.0003,
        "note": "from the first real Section B run -- kept here for the before/after record",
    },
}

print(f"\n{'='*60}\nFINAL Grad-CAM++ METRIC (target_layer=features[5])\n{'='*60}")
print(f"  Ratio          : {final_summary['overall_mean_energy_in_box_ratio']:.4f} +/- {final_summary['overall_std']:.4f}")
print(f"  Chance         : {final_summary['overall_chance_baseline_area_fraction']:.4f}")
print(f"  Ratio - chance : {final_summary['overall_ratio_minus_chance']:+.4f}")
print(f"  Paired t-test  : t={t_stat:.3f}, p={t_p:.2e}")
if w_stat is not None:
    print(f"  Wilcoxon       : W={w_stat:.1f}, p={w_p:.2e}")
for cls, s in per_class_final.items():
    print(f"    {cls:<14}: diff={s['ratio_minus_chance']:+.4f}  p={s['paired_ttest_p']}  n={s['n']}")

with open(OUT_DIR / "gradcam_localization_metrics_final.json", "w") as f:
    json.dump(final_summary, f, indent=2)
print(f"\nSaved -> {OUT_DIR / 'gradcam_localization_metrics_final.json'}")
print("\nThis is the number for the manuscript -- supersedes gradcam_localization_metrics.json "
      "(computed with the un-corrected features[-1] layer).")


Final metric (features[5]):   0%|          | 0/429 [00:00<?, ?it/s]


FINAL Grad-CAM++ METRIC (target_layer=features[5])
  Ratio          : 0.8831 +/- 0.0665
  Chance         : 0.8346
  Ratio - chance : +0.0485
  Paired t-test  : t=13.580, p=1.40e-33
  Wilcoxon       : W=7762.0, p=1.39e-28
    gametocyte    : diff=+0.0660  p=0.0001  n=27
    ring          : diff=+0.0334  p=0.0026  n=56
    trophozoite   : diff=+0.0472  p=0.0  n=221
    schizont      : diff=+0.0791  p=0.0  n=22

Saved -> /kaggle/working/phase6_outputs/gradcam_localization_metrics_final.json

This is the number for the manuscript -- supersedes gradcam_localization_metrics.json (computed with the un-corrected features[-1] layer).


---
## Plots -- rendered inline and saved as PNGs

Reads back whatever's saved in `/kaggle/working/phase6_outputs/` (same files the Appendix cell
reads) and generates a figure per section. Each one both **displays inline** (so it's part of the
notebook's own saved output -- survives even if the working directory gets wiped before you
download it) and **saves a PNG** to `phase6_outputs/plots/` (a direct, downloadable file as a
second line of defense). Safe to re-run any time -- skips whatever hasn't been produced yet.


In [16]:
# ---- Plots: inline display + saved PNGs, built from the saved JSON results ----
PLOTS_DIR = OUT_DIR / "plots"
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

def _load(fname):
    fpath = OUT_DIR / fname
    if not fpath.exists():
        print(f"  (skipping -- {fname} not found yet)")
        return None
    with open(fpath) as f:
        return json.load(f)

# --- Plot 1: Section A -- per-fold accuracy vs. single-split baseline ---
kfold_data = _load("kfold_metrics.json")
if kfold_data and kfold_data.get("fold_results"):
    folds = kfold_data["fold_results"]
    fold_ids = [r["fold"] for r in folds]
    accs = [r["best_val_acc"] for r in folds]
    mean_acc, std_acc = kfold_data.get("mean_val_acc"), kfold_data.get("std_val_acc")

    fig, ax = plt.subplots(figsize=(6, 4))
    bars = ax.bar([f"Fold {i}" for i in fold_ids], accs, color="#4C72B0")
    for b, a in zip(bars, accs):
        ax.text(b.get_x() + b.get_width() / 2, a + 0.05, f"{a:.2f}%", ha="center", fontsize=9)
    if mean_acc is not None:
        ax.axhline(mean_acc, color="#4C72B0", linestyle="--", linewidth=1,
                    label=f"{len(folds)}-fold mean = {mean_acc:.2f}% +/- {std_acc:.2f}%")
    ax.axhline(98.36, color="#C44E52", linestyle=":", linewidth=1.5,
               label="single-split original = 98.36%")
    ax.set_ylabel("Validation accuracy (%)")
    ax.set_title(f"Section A -- k-fold CV ({len(folds)}/{kfold_data.get('n_folds', '?')} folds)")
    ax.set_ylim(min(accs) - 2, 100)
    ax.legend(fontsize=8, loc="lower right")
    fig.tight_layout()
    fig.savefig(PLOTS_DIR / "kfold_accuracy.png", dpi=150)
    plt.show()
    plt.close(fig)
else:
    print("Skipping Section A plot -- no kfold_metrics.json yet.")

# --- Plot 2: Section B FINAL -- ratio vs. chance per class (features[5]) ---
gc_final = _load("gradcam_localization_metrics_final.json") or _load("gradcam_localization_metrics.json")
if gc_final and gc_final.get("per_class"):
    classes = list(gc_final["per_class"].keys())
    ratios = [gc_final["per_class"][c]["mean_ratio"] for c in classes]
    chances = [gc_final["per_class"][c]["mean_chance"] for c in classes]

    x = np.arange(len(classes))
    width = 0.35
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.bar(x - width / 2, ratios, width, label="Grad-CAM++ ratio", color="#4C72B0")
    ax.bar(x + width / 2, chances, width, label="Chance baseline", color="#C44E52")
    ax.set_xticks(x); ax.set_xticklabels(classes, rotation=20)
    ax.set_ylabel("Energy-in-box ratio")
    layer_used = gc_final.get("target_layer", "features[-1] (default)")
    ax.set_title(f"Section B -- Grad-CAM++ vs. chance per class\n({layer_used})", fontsize=10)
    ax.legend(fontsize=8)
    fig.tight_layout()
    fig.savefig(PLOTS_DIR / "gradcam_ratio_vs_chance.png", dpi=150)
    plt.show()
    plt.close(fig)
else:
    print("Skipping Section B ratio-vs-chance plot -- no data yet.")

# --- Plot 3: Section B -- layer ablation (ratio - chance per candidate layer) ---
ablation_data = _load("gradcam_layer_ablation.json")
if ablation_data:
    names = list(ablation_data.keys())
    diffs = [ablation_data[n]["ratio_minus_chance"] for n in names]
    colors = ["#55A868" if d > 0 else "#C44E52" for d in diffs]

    fig, ax = plt.subplots(figsize=(7, 4))
    bars = ax.barh(names, diffs, color=colors)
    ax.axvline(0, color="black", linewidth=0.8)
    for b, d in zip(bars, diffs):
        ax.text(d + (0.002 if d >= 0 else -0.002), b.get_y() + b.get_height() / 2,
                f"{d:+.4f}", va="center", ha="left" if d >= 0 else "right", fontsize=8)
    ax.set_xlabel("Ratio - chance (higher = better than geometric baseline)")
    ax.set_title("Section B -- Grad-CAM++ layer ablation")
    fig.tight_layout()
    fig.savefig(PLOTS_DIR / "layer_ablation.png", dpi=150)
    plt.show()
    plt.close(fig)
else:
    print("Skipping layer ablation plot -- no data yet.")

# --- Plot 4: Section C -- U-Net's own localization metrics ---
unet_data = _load("unet_stage1_metrics.json")
if unet_data:
    metric_labels = ["Recall@IoU0.5", "Precision@IoU0.5", "F1@IoU0.5",
                      "Centroid-in-box", "Bio-localization", "Infected sens.@0.5"]
    metric_vals = [
        unet_data.get("recall_at_iou50"), unet_data.get("precision_at_iou50"),
        unet_data.get("f1_at_iou50"), unet_data.get("centroid_in_box_recall"),
        unet_data.get("bio_localization_recall"),
        (unet_data.get("infected_cell_sensitivity", {}) or {}).get("iou50"),
    ]
    fig, ax = plt.subplots(figsize=(7, 4))
    bars = ax.bar(metric_labels, metric_vals, color="#8172B2")
    for b, v in zip(bars, metric_vals):
        if v is not None:
            ax.text(b.get_x() + b.get_width() / 2, v + 0.01, f"{v:.3f}", ha="center", fontsize=8)
    ax.set_ylim(0, 1)
    ax.set_ylabel("Score")
    ax.set_title("Section C -- U-Net baseline metrics")
    plt.xticks(rotation=20, ha="right")
    fig.tight_layout()
    fig.savefig(PLOTS_DIR / "unet_metrics.png", dpi=150)
    plt.show()
    plt.close(fig)
else:
    print("Skipping Section C plot -- no data yet.")

print(f"\nPNG copies saved to {PLOTS_DIR} regardless of whether they also rendered inline above.")



PNG copies saved to /kaggle/working/phase6_outputs/plots regardless of whether they also rendered inline above.


---
## Section A RERUN -- 5-fold (paste this next, after Finalize + first Plots)

Identical code to Section A above, just placed here so it can be pasted as a new cell at the end of your live session instead of scrolling back. It will detect that the existing `kfold_metrics.json` is from a different `N_FOLDS` (3), back it up automatically to `kfold_metrics_3fold_backup.json`, and start all 5 folds fresh -- this is the long one (hours), run it after you've already generated the "before" plots.


In [ ]:
from sklearn.model_selection import StratifiedKFold

_TRAIN_COUNTS = {1: 77420, 2: 1473, 3: 353, 4: 179, 5: 144, 6: 103}

def compute_focal_alpha(num_classes=NUM_CLASSES):
    total = sum(_TRAIN_COUNTS.values())
    alpha = [0.0]
    for i in range(1, num_classes):
        count = _TRAIN_COUNTS.get(i, 1)
        alpha.append(total / (len(_TRAIN_COUNTS) * count))
    s = sum(alpha[1:])
    alpha = [a / s * (num_classes - 1) for a in alpha]
    return torch.tensor(alpha, dtype=torch.float32)


class FocalLoss(nn.Module):
    def __init__(self, alpha, gamma=2.0):
        super().__init__()
        self.register_buffer("alpha", alpha)
        self.gamma = gamma

    def forward(self, logits, targets):
        ce = F.cross_entropy(logits, targets, reduction="none")
        pt = torch.exp(-ce)
        at = self.alpha[targets]
        return (at * (1.0 - pt) ** self.gamma * ce).mean()


def build_efficientnet_b0(num_classes=NUM_CLASSES, pretrained=True):
    weights = EfficientNet_B0_Weights.IMAGENET1K_V1 if pretrained else None
    model = efficientnet_b0(weights=weights)
    in_feat = model.classifier[1].in_features
    model.classifier[1] = nn.Linear(in_feat, num_classes)
    return model


def maybe_data_parallel(model):
    """NOT called anywhere in this notebook, and not recommended: a real run
    that used nn.DataParallel across a Kaggle T4x2 session's 2 GPUs hung
    completely after epoch 12 (confirmed -- 9+ hours with zero progress, same
    log line repeating). This is a known failure mode (multi-GPU scatter/
    gather deadlocking on P2P communication in Kaggle's container) on top of
    the overhead problem (a model this small on 64x64 crops doesn't have
    enough compute per batch to make splitting across 2 GPUs worth the sync
    cost anyway). Left defined only for reference -- do not call this."""
    if torch.cuda.device_count() > 1:
        return nn.DataParallel(model)
    return model


def make_loader(dataset, batch_size, shuffle, num_workers=4):
    """Shared DataLoader factory. num_workers>0 is safe again now that crop
    metadata is numpy-backed (see MalariaCropDataset docstring) -- the previous
    OOM was from a Python list-of-dicts, not from having workers per se. Workers
    let CPU-side augmentation (PIL crop/rotate/color-jitter) overlap with GPU
    compute instead of serializing with it, which is the main lever left after
    fixing the pixel-cache I/O and the RAM leak."""
    kwargs = dict(batch_size=batch_size, shuffle=shuffle, pin_memory=True)
    if num_workers > 0:
        kwargs.update(num_workers=num_workers, persistent_workers=True, prefetch_factor=4)
    else:
        kwargs["num_workers"] = 0
    return DataLoader(dataset, **kwargs)


def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for crops, labels in loader:
        crops, labels = crops.to(device), labels.to(device)
        logits = model(crops)
        loss = criterion(logits, labels)
        optimizer.zero_grad(); loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()
        total_loss += loss.item() * len(labels)
        correct += (logits.argmax(1) == labels).sum().item()
        total += len(labels)
    return total_loss / total, correct / total


@torch.no_grad()
def evaluate_fold(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    class_correct = {i: 0 for i in range(NUM_CLASSES)}
    class_total = {i: 0 for i in range(NUM_CLASSES)}
    for crops, labels in loader:
        crops, labels = crops.to(device), labels.to(device)
        logits = model(crops)
        loss = criterion(logits, labels)
        preds = logits.argmax(1)
        total_loss += loss.item() * len(labels)
        correct += (preds == labels).sum().item()
        total += len(labels)
        for lbl, pred in zip(labels.cpu(), preds.cpu()):
            class_total[lbl.item()] += 1
            class_correct[lbl.item()] += int(pred == lbl)
    per_class_acc = {INT_TO_LABEL[i]: round(class_correct[i] / class_total[i] * 100, 2)
                      for i in range(NUM_CLASSES) if class_total[i] > 0}
    return total_loss / total, correct / total, per_class_acc


N_FOLDS  = 5             # the TRUE k-fold structure -- keep this fixed at 5 across every
                         # session. StratifiedKFold's fold boundaries depend on n_splits, so
                         # changing N_FOLDS between runs (e.g. 3 now, 5 later) would define a
                         # DIFFERENT partition and make old "fold 1/2/3" incompatible with new
                         # ones -- not a real 5-fold CV, just two unrelated experiments glued
                         # together. To split the work across sessions without that problem,
                         # leave N_FOLDS=5 always and only change MAX_FOLDS_THIS_SESSION below.
MAX_FOLDS_THIS_SESSION = 5  # doing a full fresh 5-fold run this time (not splitting across
                            # sessions) -- set lower again later if GPU quota gets tight mid-run.
N_EPOCHS = 30            # matches the paper's original Stage 2 run (best epoch was 27/30)
EARLY_STOP_PATIENCE = 6  # stop a fold early if val_acc hasn't improved in this many epochs
BATCH    = 64
LR       = 1e-4

all_labels = np.array([full_crop_ds.label_of(i) for i in range(len(full_crop_ds))])
all_label_idx = np.array([LABEL_TO_INT[l] for l in all_labels])

# StratifiedKFold (not plain KFold): with schizont (179), gametocyte (144), and
# leukocyte (103) this rare relative to 77,420 RBCs, a random KFold can easily
# starve a fold of a whole class. Stratifying on label keeps class proportions
# consistent across all 5 folds/val splits.
kf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

# Built ONCE outside the fold loop -- previously this was rebuilt from JSON
# inside the loop on every fold, which (before the pixel-cache fix above)
# re-triggered a full disk decode 5 times over. Now it's a free shared view.
val_ds_noaug = full_crop_ds.with_mode(train=False)

# RESUME SUPPORT: nothing used to be saved until ALL folds finished -- if the
# kernel got killed by a GPU-quota/session-time limit mid-run, every completed
# fold's work was lost with it. Now every fold is saved to disk immediately
# after it finishes, and already-completed folds (matched by fold number) are
# skipped on re-run so you don't burn quota redoing them.
ckpt_path = OUT_DIR / "kfold_metrics.json"
if ckpt_path.exists():
    with open(ckpt_path) as f:
        _ckpt = json.load(f)
    _ckpt_n_folds = _ckpt.get("n_folds")
    if _ckpt_n_folds != N_FOLDS:
        # SAFETY CHECK: a checkpoint from a DIFFERENT n_splits is not resumable --
        # StratifiedKFold's fold boundaries depend on n_splits, so "fold 1" of a
        # 3-way split is different data than "fold 1" of a 5-way split. Silently
        # resuming here would splice two incompatible experiments together and
        # report a fake "5-fold" result that's actually 3-fold + 2-fold glued
        # together. Back up the old file instead of overwriting it (it's still a
        # valid standalone N_FOLDS={_ckpt_n_folds} result) and start clean.
        backup_path = OUT_DIR / f"kfold_metrics_{_ckpt_n_folds}fold_backup.json"
        ckpt_path.rename(backup_path)
        print(f"Checkpoint at {ckpt_path} was from a different N_FOLDS "
              f"({_ckpt_n_folds}, now running {N_FOLDS}) -- NOT resumable (different "
              f"StratifiedKFold partition). Backed up old result -> {backup_path}. "
              f"Starting all {N_FOLDS} folds fresh.")
        fold_results, completed_folds = [], set()
    else:
        fold_results = _ckpt.get("fold_results", [])
        completed_folds = {r["fold"] for r in fold_results}
        print(f"Resuming from {ckpt_path}: {len(completed_folds)} fold(s) already completed "
              f"({sorted(completed_folds)}) -- these will be skipped.")
else:
    fold_results, completed_folds = [], set()

t_start = time.time()

folds_started_this_session = 0

for fold, (train_idx, val_idx) in enumerate(kf.split(np.arange(len(full_crop_ds)), all_label_idx), start=1):
    if fold in completed_folds:
        continue
    if folds_started_this_session >= MAX_FOLDS_THIS_SESSION:
        print(f"\nReached MAX_FOLDS_THIS_SESSION={MAX_FOLDS_THIS_SESSION} for this session "
              f"({len(fold_results)}/{N_FOLDS} folds done overall). Re-run this cell in a later "
              f"session (raise MAX_FOLDS_THIS_SESSION, or set it >= {N_FOLDS}) to do the rest -- "
              f"N_FOLDS is unchanged so the remaining folds are still the correct ones.")
        break
    folds_started_this_session += 1
    print(f"\n{'='*60}\nFold {fold}/{N_FOLDS}  (train={len(train_idx)}  val={len(val_idx)})\n{'='*60}")

    train_subset = Subset(full_crop_ds, train_idx.tolist())
    val_subset = Subset(val_ds_noaug, val_idx.tolist())

    train_loader = make_loader(train_subset, BATCH, shuffle=True)
    val_loader   = make_loader(val_subset,   BATCH, shuffle=False)

    model = build_efficientnet_b0().to(device)  # no DataParallel here by default -- see note above
    alpha = compute_focal_alpha().to(device)
    criterion = FocalLoss(alpha=alpha, gamma=2.0)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=N_EPOCHS)

    best_val_acc, best_per_class, best_epoch, epochs_since_improve = 0.0, {}, 0, 0
    best_smoothed, val_acc_history = 0.0, []
    fold_start = time.time()
    for epoch in range(N_EPOCHS):
        tr_loss, tr_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
        vl_loss, vl_acc, per_class = evaluate_fold(model, val_loader, criterion, device)
        scheduler.step()
        val_acc_history.append(vl_acc)

        # Raw best (for reporting -- comparable to the paper's single-checkpoint number)
        if vl_acc > best_val_acc + 1e-4:
            best_val_acc, best_per_class, best_epoch = vl_acc, per_class, epoch + 1

        # Smoothed (3-epoch moving average) for the STOP decision -- val_acc swings
        # epoch to epoch (e.g. 90%->96%->91% seen in the first real run), so patience
        # on the raw number either resets on noise or never triggers. The smoothed
        # trend is what actually tells you whether the fold is still improving.
        smoothed = float(np.mean(val_acc_history[-3:]))
        if smoothed > best_smoothed + 1e-4:
            best_smoothed, epochs_since_improve = smoothed, 0
        else:
            epochs_since_improve += 1

        ep_time = time.time() - fold_start
        print(f"  epoch {epoch+1:02d}/{N_EPOCHS}  train_acc={tr_acc*100:.2f}%  val_acc={vl_acc*100:.2f}%  "
              f"(smoothed={smoothed*100:.2f}%, {ep_time/(epoch+1):.0f}s/epoch avg)")
        if epochs_since_improve >= EARLY_STOP_PATIENCE:
            print(f"  Early stop: smoothed val_acc hasn't improved in {EARLY_STOP_PATIENCE} epochs "
                  f"(best raw {best_val_acc*100:.2f}% at epoch {best_epoch}).")
            break

    n_epochs_run = epoch + 1
    plateaued = best_epoch <= n_epochs_run - 3 or n_epochs_run < N_EPOCHS
    print(f"  Fold {fold} best val acc: {best_val_acc*100:.2f}% (epoch {best_epoch}/{n_epochs_run} run)"
          f"  -- {'plateaued' if plateaued else 'STILL IMPROVING at cutoff, consider more epochs'}")
    fold_results.append({
        "fold": fold, "best_val_acc": round(best_val_acc * 100, 2), "best_epoch": best_epoch,
        "n_epochs_run": n_epochs_run, "plateaued": plateaued, "per_class_acc": best_per_class,
        "val_acc_history": [round(a * 100, 2) for a in val_acc_history],
    })

    # Explicit cleanup -- each fold builds a fresh model/optimizer/scheduler/loaders;
    # without deleting them, GPU memory and host RAM both accumulate fold over fold
    # instead of being reclaimed, which compounds the DataLoader-worker RAM issue above.
    del model, optimizer, scheduler, train_loader, val_loader, train_subset, val_subset
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    # Save after EVERY fold -- if this is the last one that finishes before the
    # session gets killed, you still have real numbers for however many folds
    # completed instead of nothing at all.
    _partial = {
        "n_folds": N_FOLDS, "n_epochs_cap": N_EPOCHS, "early_stop_patience": EARLY_STOP_PATIENCE,
        "kfold_strategy": "StratifiedKFold", "fold_results": fold_results,
        "n_folds_completed": len(fold_results),
        "status": "in_progress" if len(fold_results) < N_FOLDS else "complete",
    }
    with open(ckpt_path, "w") as f:
        json.dump(_partial, f, indent=2)
    print(f"  Checkpoint saved ({len(fold_results)}/{N_FOLDS} folds done) -> {ckpt_path}")

elapsed = time.time() - t_start
accs = np.array([r["best_val_acc"] for r in fold_results])
n_not_plateaued = sum(1 for r in fold_results if not r["plateaued"])

kfold_summary = {
    "n_folds": N_FOLDS, "n_epochs_cap": N_EPOCHS, "early_stop_patience": EARLY_STOP_PATIENCE,
    "kfold_strategy": "StratifiedKFold",
    "fold_results": fold_results,
    "n_folds_completed": len(fold_results),
    "status": "complete" if len(fold_results) == N_FOLDS else "PARTIAL -- ran out of folds before finishing",
    "mean_val_acc": round(float(accs.mean()), 2),
    "std_val_acc":  round(float(accs.std()), 2),
    "n_folds_not_plateaued": n_not_plateaued,
    "elapsed_seconds": round(elapsed, 1),
}

print(f"\n{'='*60}\nK-FOLD SUMMARY ({len(fold_results)}/{N_FOLDS} folds)\n{'='*60}")
print(f"  Accuracy across {len(fold_results)} fold(s): {kfold_summary['mean_val_acc']:.2f}% "
      f"+/- {kfold_summary['std_val_acc']:.2f}%")
print(f"  (single-split original figure was 98.36%, best epoch 27/30)")
if len(fold_results) < N_FOLDS:
    print(f"  NOTE: only {len(fold_results)}/{N_FOLDS} folds completed -- re-run this cell (it will "
          f"resume and skip completed folds) to finish before trusting this for the paper.")
if n_not_plateaued:
    print(f"  WARNING: {n_not_plateaued} completed fold(s) hit the {N_EPOCHS}-epoch cap while still "
          f"improving -- re-run with a higher N_EPOCHS before trusting this number for the paper.")

with open(ckpt_path, "w") as f:
    json.dump(kfold_summary, f, indent=2)
print(f"\nSaved -> {ckpt_path}")


---
## Plots (again) -- after the 5-fold rerun

Same Plots cell as above, pasted again so it can run after Section A's 5-fold rerun finishes -- it reads whatever's on disk at the time it runs, so this second pass picks up the new 5-fold `kfold_metrics.json` automatically and regenerates the Section A figure with the updated numbers (Sections B/C plots will look the same as the first pass, since those aren't touched by the rerun).


In [ ]:
# ---- Plots: inline display + saved PNGs, built from the saved JSON results ----
PLOTS_DIR = OUT_DIR / "plots"
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

def _load(fname):
    fpath = OUT_DIR / fname
    if not fpath.exists():
        print(f"  (skipping -- {fname} not found yet)")
        return None
    with open(fpath) as f:
        return json.load(f)

# --- Plot 1: Section A -- per-fold accuracy vs. single-split baseline ---
kfold_data = _load("kfold_metrics.json")
if kfold_data and kfold_data.get("fold_results"):
    folds = kfold_data["fold_results"]
    fold_ids = [r["fold"] for r in folds]
    accs = [r["best_val_acc"] for r in folds]
    mean_acc, std_acc = kfold_data.get("mean_val_acc"), kfold_data.get("std_val_acc")

    fig, ax = plt.subplots(figsize=(6, 4))
    bars = ax.bar([f"Fold {i}" for i in fold_ids], accs, color="#4C72B0")
    for b, a in zip(bars, accs):
        ax.text(b.get_x() + b.get_width() / 2, a + 0.05, f"{a:.2f}%", ha="center", fontsize=9)
    if mean_acc is not None:
        ax.axhline(mean_acc, color="#4C72B0", linestyle="--", linewidth=1,
                    label=f"{len(folds)}-fold mean = {mean_acc:.2f}% +/- {std_acc:.2f}%")
    ax.axhline(98.36, color="#C44E52", linestyle=":", linewidth=1.5,
               label="single-split original = 98.36%")
    ax.set_ylabel("Validation accuracy (%)")
    ax.set_title(f"Section A -- k-fold CV ({len(folds)}/{kfold_data.get('n_folds', '?')} folds)")
    ax.set_ylim(min(accs) - 2, 100)
    ax.legend(fontsize=8, loc="lower right")
    fig.tight_layout()
    fig.savefig(PLOTS_DIR / "kfold_accuracy.png", dpi=150)
    plt.show()
    plt.close(fig)
else:
    print("Skipping Section A plot -- no kfold_metrics.json yet.")

# --- Plot 2: Section B FINAL -- ratio vs. chance per class (features[5]) ---
gc_final = _load("gradcam_localization_metrics_final.json") or _load("gradcam_localization_metrics.json")
if gc_final and gc_final.get("per_class"):
    classes = list(gc_final["per_class"].keys())
    ratios = [gc_final["per_class"][c]["mean_ratio"] for c in classes]
    chances = [gc_final["per_class"][c]["mean_chance"] for c in classes]

    x = np.arange(len(classes))
    width = 0.35
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.bar(x - width / 2, ratios, width, label="Grad-CAM++ ratio", color="#4C72B0")
    ax.bar(x + width / 2, chances, width, label="Chance baseline", color="#C44E52")
    ax.set_xticks(x); ax.set_xticklabels(classes, rotation=20)
    ax.set_ylabel("Energy-in-box ratio")
    layer_used = gc_final.get("target_layer", "features[-1] (default)")
    ax.set_title(f"Section B -- Grad-CAM++ vs. chance per class\n({layer_used})", fontsize=10)
    ax.legend(fontsize=8)
    fig.tight_layout()
    fig.savefig(PLOTS_DIR / "gradcam_ratio_vs_chance.png", dpi=150)
    plt.show()
    plt.close(fig)
else:
    print("Skipping Section B ratio-vs-chance plot -- no data yet.")

# --- Plot 3: Section B -- layer ablation (ratio - chance per candidate layer) ---
ablation_data = _load("gradcam_layer_ablation.json")
if ablation_data:
    names = list(ablation_data.keys())
    diffs = [ablation_data[n]["ratio_minus_chance"] for n in names]
    colors = ["#55A868" if d > 0 else "#C44E52" for d in diffs]

    fig, ax = plt.subplots(figsize=(7, 4))
    bars = ax.barh(names, diffs, color=colors)
    ax.axvline(0, color="black", linewidth=0.8)
    for b, d in zip(bars, diffs):
        ax.text(d + (0.002 if d >= 0 else -0.002), b.get_y() + b.get_height() / 2,
                f"{d:+.4f}", va="center", ha="left" if d >= 0 else "right", fontsize=8)
    ax.set_xlabel("Ratio - chance (higher = better than geometric baseline)")
    ax.set_title("Section B -- Grad-CAM++ layer ablation")
    fig.tight_layout()
    fig.savefig(PLOTS_DIR / "layer_ablation.png", dpi=150)
    plt.show()
    plt.close(fig)
else:
    print("Skipping layer ablation plot -- no data yet.")

# --- Plot 4: Section C -- U-Net's own localization metrics ---
unet_data = _load("unet_stage1_metrics.json")
if unet_data:
    metric_labels = ["Recall@IoU0.5", "Precision@IoU0.5", "F1@IoU0.5",
                      "Centroid-in-box", "Bio-localization", "Infected sens.@0.5"]
    metric_vals = [
        unet_data.get("recall_at_iou50"), unet_data.get("precision_at_iou50"),
        unet_data.get("f1_at_iou50"), unet_data.get("centroid_in_box_recall"),
        unet_data.get("bio_localization_recall"),
        (unet_data.get("infected_cell_sensitivity", {}) or {}).get("iou50"),
    ]
    fig, ax = plt.subplots(figsize=(7, 4))
    bars = ax.bar(metric_labels, metric_vals, color="#8172B2")
    for b, v in zip(bars, metric_vals):
        if v is not None:
            ax.text(b.get_x() + b.get_width() / 2, v + 0.01, f"{v:.3f}", ha="center", fontsize=8)
    ax.set_ylim(0, 1)
    ax.set_ylabel("Score")
    ax.set_title("Section C -- U-Net baseline metrics")
    plt.xticks(rotation=20, ha="right")
    fig.tight_layout()
    fig.savefig(PLOTS_DIR / "unet_metrics.png", dpi=150)
    plt.show()
    plt.close(fig)
else:
    print("Skipping Section C plot -- no data yet.")

print(f"\nPNG copies saved to {PLOTS_DIR} regardless of whether they also rendered inline above.")
